# RM2 Actor Type, Target, Goals, and Observed Association Typology

Notebook ini memperbaiki pemetaan tiga actor type untuk RM2 dengan universe aktor final yang dibangun dari
akun komentator, pembuat video pada metadata, dan registry individual yang terverifikasi.


## 2. Kerangka tiga actor type

Actor type utama dibatasi pada tiga kategori:

- `Individual Actor`: akun pembuat video yang tercatat pada metadata video, serta akun individual tambahan
  yang telah diverifikasi melalui registry.
- `Community Actor`: akun anggota HCC final, selama akun tersebut bukan Individual Actor.
- `Mass Actor`: akun komentator yang bukan Individual Actor dan bukan anggota HCC.

Metrik centrality, LCN membership, direct interaction, temporal association, dan shared-video context
digunakan sebagai atribut sekunder, bukan actor type utama.


## 3. Batasan interpretasi

Hubungan antara Individual Actor, Community Actor, dan Mass Actor dalam notebook ini merupakan hubungan
struktural, interaksional, temporal, atau konteks video yang teramati. Hubungan tersebut tidak membuktikan
pengaruh kausal, perubahan opini, niat manipulatif, ataupun astroturfing.

Istilah yang digunakan adalah observed association, observed interaction, temporal association,
shared-video context, konteks aktivitas yang sama, keselarasan sentimen yang teramati, dan indikasi
orientasi pesan.


In [1]:
# 4. Import dan konfigurasi
from __future__ import annotations

import hashlib
import math
import re
import shutil
from decimal import Decimal, InvalidOperation
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
DATASET_PATH = ROOT / "dataset.csv"
METADATA_PATH = ROOT / "video_metadata_clean.csv"
REGISTRY_PATH = ROOT / "config" / "individual_actor_registry.csv"
LCN_NODES_PATH = ROOT / "output" / "gephi" / "gephi_lcn_nodes.csv"
HCC_NODES_PATH = ROOT / "output" / "gephi" / "gephi_hcc_nodes.csv"
HCC_EDGES_PATH = ROOT / "output" / "gephi" / "gephi_hcc_edges.csv"
COMMENT_SENTIMENT_PATH = ROOT / "output" / "rm2_sentiment" / "tables" / "comment_sentiment.csv"
ACCOUNT_SENTIMENT_PATH = ROOT / "output" / "rm2_sentiment" / "tables" / "account_sentiment_summary.csv"
HCC_SENTIMENT_PATH = ROOT / "output" / "rm2_sentiment" / "tables" / "hcc_sentiment_goals_summary.csv"

OUT_DIR = ROOT / "output" / "rm2_actor_type"
TABLE_DIR = OUT_DIR / "tables"
VIS_DIR = OUT_DIR / "visualisasi"
GEPHI_DIR = OUT_DIR / "gephi"

ACCOUNT_GOAL_MIN_COMMENTS = 3
MASS_TEMPORAL_WINDOW_MINUTES = 60
ANONYMIZE_PUBLIC_MASS_ACTORS = True
MASS_HASH_SALT = "rm2_actor_type_public_mass_hash_v1"

EXPECTED_BASELINES = {
    "dataset_comments": 33847,
    "account_sentiment_rows": 26424,
    "lcn_nodes": 724,
    "hcc_nodes": 218,
    "hcc_communities": 42,
    "hcc_edges": 464,
}

REGISTRY_COLUMNS = [
    "username",
    "display_name",
    "individual_subtype",
    "associated_brand",
    "identification_source",
    "verification_status",
    "notes",
]

VALID_ACTOR_TYPES = ["Individual Actor", "Community Actor", "Mass Actor"]
VALID_HCC_ASSOCIATION_STATUS = [
    "Unique Direct Association",
    "Unique Temporal Association",
    "Single Shared-Video Context",
    "Multiple / Ambiguous HCC",
    "No HCC Association",
]
VALID_EXPOSURE_LEVELS = [
    "Direct Interaction",
    "Temporal Association",
    "Shared-Video Context Only",
    "No Observed Community Association",
]
VALID_ALIGNMENT_VALUES = [
    "Aligned",
    "Not aligned",
    "Mixed / Inconclusive",
    "Insufficient Text",
    "Ambiguous HCC Association",
    "Context Only",
    "No HCC Association",
]

def safe_clean_output_dir(path: Path) -> None:
    resolved_root = ROOT.resolve()
    resolved_path = path.resolve()
    if resolved_path == resolved_root or resolved_root not in resolved_path.parents:
        raise ValueError(f"Refusing to clean unsafe path: {resolved_path}")
    if path.exists():
        shutil.rmtree(path)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    VIS_DIR.mkdir(parents=True, exist_ok=True)
    GEPHI_DIR.mkdir(parents=True, exist_ok=True)

safe_clean_output_dir(OUT_DIR)
REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
if not REGISTRY_PATH.exists():
    pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(REGISTRY_PATH, index=False)


In [2]:
# 5. Audit input dan checksum
PROTECTED_INPUTS = {
    "dataset": DATASET_PATH,
    "metadata": METADATA_PATH,
    "rm1_notebook": ROOT / "tiktok_coordination_analysis.ipynb",
    "rm2_sentiment_notebook": ROOT / "02_rm2_sentiment_goals.ipynb",
    "lcn_nodes": LCN_NODES_PATH,
    "hcc_nodes": HCC_NODES_PATH,
    "hcc_edges": HCC_EDGES_PATH,
    "comment_sentiment": COMMENT_SENTIMENT_PATH,
    "account_sentiment": ACCOUNT_SENTIMENT_PATH,
    "hcc_sentiment": HCC_SENTIMENT_PATH,
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

baseline_checksums = {name: sha256_file(path) for name, path in PROTECTED_INPUTS.items()}

dataset = pd.read_csv(DATASET_PATH, dtype=str, low_memory=False)
video_metadata = pd.read_csv(METADATA_PATH, dtype=str, low_memory=False)
lcn_nodes = pd.read_csv(LCN_NODES_PATH, dtype=str, low_memory=False)
hcc_nodes = pd.read_csv(HCC_NODES_PATH, dtype=str, low_memory=False)
hcc_edges = pd.read_csv(HCC_EDGES_PATH, dtype=str, low_memory=False)
comment_sentiment = pd.read_csv(COMMENT_SENTIMENT_PATH, dtype=str, low_memory=False)
account_sentiment = pd.read_csv(ACCOUNT_SENTIMENT_PATH, dtype=str, low_memory=False)
hcc_sentiment = pd.read_csv(HCC_SENTIMENT_PATH, dtype=str, low_memory=False)

baseline_audit = {
    "dataset_comments": len(dataset),
    "account_sentiment_rows": len(account_sentiment),
    "lcn_nodes": len(lcn_nodes),
    "hcc_nodes": len(hcc_nodes),
    "hcc_communities": hcc_nodes["community"].nunique(),
    "hcc_edges": len(hcc_edges),
}

for key, expected in EXPECTED_BASELINES.items():
    actual = baseline_audit[key]
    if actual != expected:
        raise AssertionError(f"Baseline mismatch for {key}: expected {expected}, got {actual}")

print("Baseline audit:")
for key, value in baseline_audit.items():
    print(f"- {key}: {value}")
print("Checksum sample:")
for key in ["dataset", "comment_sentiment", "hcc_nodes"]:
    print(f"- {key}: {baseline_checksums[key]}")


Baseline audit:
- dataset_comments: 33847
- account_sentiment_rows: 26424
- lcn_nodes: 724
- hcc_nodes: 218
- hcc_communities: 42
- hcc_edges: 464
Checksum sample:
- dataset: 7e0ea9cbc82243445cccea1a14ccb847c61235e932f754f5e1bc42e390df32da
- comment_sentiment: 8c27dadcab3e2723cf9187dd7214c0091cfe13d1333c6860ff702a47f0b4dfd2
- hcc_nodes: a87aedc3eac4d1ad2ea89bcbb5d9b9156c70aa20e26df5ad20946ab18b840bb0


In [3]:
# 6. Normalisasi username dan ID
def normalize_username(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = text.lstrip("@")
    text = re.sub(r"\s+", "", text)
    return text

def normalize_text_id(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null", "<na>"}:
        return ""
    if re.fullmatch(r"\d+\.0", text):
        return text[:-2]
    if re.fullmatch(r"\d+", text):
        return text
    if re.fullmatch(r"\d+(?:\.\d+)?e[+-]?\d+", text, flags=re.I):
        try:
            return format(Decimal(text).to_integral_value(), "f")
        except (InvalidOperation, ValueError):
            return text
    return text

def float_round_key(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not re.fullmatch(r"\d+", text):
        return ""
    try:
        return format(Decimal(str(float(text))).to_integral_value(), "f")
    except (InvalidOperation, ValueError, OverflowError):
        return ""

def extract_creator_from_url(value) -> str:
    if pd.isna(value):
        return ""
    match = re.search(r"/@([^/]+)/", str(value))
    return normalize_username(match.group(1)) if match else ""

def to_numeric(series, fill_value=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(fill_value)

def numeric_or_nan(series):
    return pd.to_numeric(series, errors="coerce")

def product_category_to_brand(value) -> str:
    text = str(value).strip().lower()
    if "azarine" in text:
        return "Azarine"
    if "daviena" in text or "davina" in text:
        return "Daviena/Davina"
    if "maryame" in text:
        return "Maryame"
    if "originote" in text:
        return "The Originote"
    if text in {"", "nan", "none"}:
        return "Unidentified"
    return str(value).strip()

def public_mass_id(username_norm: str) -> str:
    digest = hashlib.sha256(f"{MASS_HASH_SALT}:{username_norm}".encode("utf-8")).hexdigest()[:12]
    return f"MASS_{digest}"

def coverage_label(value: float) -> str:
    if pd.isna(value):
        return "Very Low Coverage"
    if value >= 0.75:
        return "High Coverage"
    if value >= 0.40:
        return "Medium Coverage"
    if value >= 0.10:
        return "Low Coverage"
    return "Very Low Coverage"

def clean_segment(value: str) -> str:
    text = str(value).replace("/", " ").replace("-", " ")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text[:80] or "NA"

def join_unique(values) -> str:
    vals = sorted({str(v) for v in values if pd.notna(v) and str(v).strip() != ""})
    return ";".join(vals)

def sentiment_goal_from_distribution(pos, neu, neg, n_valid, min_comments=ACCOUNT_GOAL_MIN_COMMENTS, allow_insufficient=True):
    pos = int(pos or 0)
    neu = int(neu or 0)
    neg = int(neg or 0)
    n_valid = int(n_valid or 0)
    if n_valid == 0:
        return "Insufficient Text", "None"
    if allow_insufficient and n_valid < min_comments:
        return "Insufficient Text", "None"
    pos_r, neu_r, neg_r = pos / n_valid, neu / n_valid, neg / n_valid
    ratios = {"Positive": pos_r, "Neutral": neu_r, "Negative": neg_r}
    dominant_sentiment = max(ratios, key=ratios.get)
    top_ratio = ratios[dominant_sentiment]
    if pos_r >= 0.35 and neg_r >= 0.35:
        goal = "Polarized / Contested"
    elif top_ratio < 0.45:
        goal = "Mixed Goals"
    elif dominant_sentiment == "Positive":
        goal = "Promotional / Supportive"
    elif dominant_sentiment == "Negative":
        goal = "Critical / Complaint"
    else:
        goal = "Neutral Engagement"
    if n_valid >= 10 and top_ratio >= 0.60:
        confidence = "High"
    elif n_valid >= 5 and top_ratio >= 0.50:
        confidence = "Medium"
    else:
        confidence = "Low"
    return goal, confidence

def dominant_sentiment_from_counts(pos, neu, neg, n_valid):
    n_valid = int(n_valid or 0)
    if n_valid == 0:
        return "Not Observed"
    values = {"Positive": int(pos or 0), "Neutral": int(neu or 0), "Negative": int(neg or 0)}
    top = max(values.values())
    winners = [k for k, v in values.items() if v == top]
    return winners[0] if len(winners) == 1 else "Mixed"


In [4]:
# 7. Build final actor universe
dataset = dataset.copy()
video_metadata = video_metadata.copy()
account_sentiment = account_sentiment.copy()
comment_sentiment = comment_sentiment.copy()
hcc_nodes = hcc_nodes.copy()
lcn_nodes = lcn_nodes.copy()
hcc_sentiment = hcc_sentiment.copy()

dataset["username_norm"] = dataset["username"].map(normalize_username)
dataset["parent_user_norm"] = dataset["parent_user"].map(normalize_username)
dataset["comment_id_norm"] = dataset["comment_id"].map(normalize_text_id)
dataset["comment_id_float_key"] = dataset["comment_id"].map(float_round_key)
dataset["parent_comment_id_norm"] = dataset["parent_comment_id"].map(normalize_text_id)
dataset["video_id_norm"] = dataset["video_id"].map(normalize_text_id)
dataset["timestamp_dt"] = pd.to_datetime(dataset["timestamp"], errors="coerce", utc=True)
dataset["video_creator_norm"] = dataset["video_url"].map(extract_creator_from_url)
dataset["product_brand"] = dataset["product_category"].map(product_category_to_brand)

comment_sentiment["username_norm"] = comment_sentiment["username"].map(normalize_username)
comment_sentiment["comment_id_norm"] = comment_sentiment["comment_id"].map(normalize_text_id)
comment_sentiment["video_id_norm"] = comment_sentiment["video_id"].map(normalize_text_id)
comment_sentiment["timestamp_dt"] = pd.to_datetime(comment_sentiment["timestamp"], errors="coerce", utc=True)
comment_sentiment["product_brand"] = comment_sentiment["product_category"].map(product_category_to_brand)

account_sentiment["username_norm"] = account_sentiment["username"].map(normalize_username)
hcc_nodes["username_norm"] = hcc_nodes["id"].map(normalize_username)
lcn_nodes["username_norm"] = lcn_nodes["id"].map(normalize_username)
video_metadata["creator_norm"] = video_metadata["creator"].map(normalize_username)
video_metadata["metadata_brand"] = video_metadata["product_category"].map(product_category_to_brand)

registry = pd.read_csv(REGISTRY_PATH, dtype=str, keep_default_na=False)
for col in REGISTRY_COLUMNS:
    if col not in registry.columns:
        registry[col] = ""
registry = registry[REGISTRY_COLUMNS].copy()
registry["username_norm"] = registry["username"].map(normalize_username)
registry["verification_status_norm"] = registry["verification_status"].astype(str).str.strip().str.lower()

commenters = sorted(set(account_sentiment["username_norm"].dropna()) - {""})
metadata_creators = sorted(set(video_metadata["creator_norm"].dropna()) - {""})
verified_registry = sorted(set(registry.loc[registry["verification_status_norm"].eq("verified"), "username_norm"]) - {""})

actor_universe_norm = sorted(set(commenters) | set(metadata_creators) | set(verified_registry))
actors = pd.DataFrame({"username_norm": actor_universe_norm})

account_cols = [
    "username",
    "username_norm",
    "n_comments",
    "n_valid_text_comments",
    "positive_count",
    "neutral_count",
    "negative_count",
    "positive_ratio",
    "neutral_ratio",
    "negative_ratio",
    "dominant_sentiment",
    "avg_sentiment_confidence",
    "is_hcc_member",
    "community",
    "primary_brand",
    "brand_label_auto",
    "brand_combo",
    "brand_confidence",
    "degree",
    "weighted_degree",
    "betweenness",
    "narrative_similarity_level",
]
actors = actors.merge(account_sentiment[account_cols], on="username_norm", how="left", suffixes=("", "_account"))
actors["username"] = actors["username"].fillna(actors["username_norm"])

numeric_zero_cols = [
    "n_comments",
    "n_valid_text_comments",
    "positive_count",
    "neutral_count",
    "negative_count",
    "degree",
    "weighted_degree",
    "betweenness",
]
for col in numeric_zero_cols:
    actors[col] = to_numeric(actors[col], fill_value=0)
for col in ["positive_ratio", "neutral_ratio", "negative_ratio", "avg_sentiment_confidence"]:
    actors[col] = numeric_or_nan(actors[col])
actors.loc[actors["n_comments"].eq(0), "dominant_sentiment"] = "Not Observed"
actors.loc[actors["n_comments"].eq(0), ["positive_ratio", "neutral_ratio", "negative_ratio"]] = np.nan
actors["is_commenter"] = actors["username_norm"].isin(commenters)
actors["is_video_creator"] = actors["username_norm"].isin(metadata_creators)
actors["is_verified_registry_actor"] = actors["username_norm"].isin(verified_registry)

hcc_lookup = hcc_nodes.set_index("username_norm")["community"].to_dict()
lcn_set = set(lcn_nodes["username_norm"].dropna()) - {""}
hcc_set = set(hcc_nodes["username_norm"].dropna()) - {""}
actors["is_hcc_member"] = actors["username_norm"].isin(hcc_set)
actors["community"] = actors["username_norm"].map(hcc_lookup).fillna(actors["community"])
actors["is_lcn_member"] = actors["username_norm"].isin(lcn_set)
actors["individual_in_hcc"] = actors["is_video_creator"] & actors["is_hcc_member"]

def classify_actor(row):
    if row["is_video_creator"] or row["is_verified_registry_actor"]:
        return "Individual Actor"
    if row["is_hcc_member"]:
        return "Community Actor"
    return "Mass Actor"

actors["actor_type_primary"] = actors.apply(classify_actor, axis=1)
actors["mass_position_secondary"] = np.where(
    actors["actor_type_primary"].eq("Mass Actor") & actors["is_lcn_member"],
    "LCN Non-HCC",
    np.where(actors["actor_type_primary"].eq("Mass Actor"), "Outside LCN", ""),
)
actors["actor_id_anonymous"] = np.where(
    actors["actor_type_primary"].eq("Mass Actor"),
    actors["username_norm"].map(public_mass_id),
    actors["username_norm"],
)
actors["public_username"] = np.where(
    ANONYMIZE_PUBLIC_MASS_ACTORS & actors["actor_type_primary"].eq("Mass Actor"),
    actors["actor_id_anonymous"],
    actors["username"],
)


In [5]:
# 8. Identifikasi video creators
creator_video_stats = (
    video_metadata.loc[video_metadata["creator_norm"].ne("")]
    .groupby("creator_norm")
    .agg(
        creator_video_count=("video_url", "nunique"),
        creator_product_categories=("product_category", join_unique),
        creator_brands=("metadata_brand", join_unique),
        creator_raw_names=("creator", join_unique),
    )
    .reset_index()
)

video_context = (
    dataset[["video_id_norm", "video_url", "video_creator_norm", "product_category", "product_brand", "source_file"]]
    .drop_duplicates()
    .copy()
)
creator_from_url_stats = (
    video_context.loc[video_context["video_creator_norm"].ne("")]
    .groupby("video_creator_norm")
    .agg(
        dataset_video_count=("video_id_norm", "nunique"),
        dataset_product_categories=("product_category", join_unique),
        dataset_brands=("product_brand", join_unique),
    )
    .reset_index()
    .rename(columns={"video_creator_norm": "creator_norm"})
)
creator_video_stats = creator_video_stats.merge(creator_from_url_stats, on="creator_norm", how="outer")
for col in ["creator_video_count", "dataset_video_count"]:
    creator_video_stats[col] = to_numeric(creator_video_stats[col], fill_value=0).astype(int)

actors = actors.merge(creator_video_stats, left_on="username_norm", right_on="creator_norm", how="left")
actors.drop(columns=["creator_norm"], inplace=True)
actors["creator_video_count"] = to_numeric(actors["creator_video_count"], fill_value=0).astype(int)
actors["dataset_video_count"] = to_numeric(actors["dataset_video_count"], fill_value=0).astype(int)

print("Video creator audit:")
print(f"- metadata creator raw rows: {video_metadata['creator'].notna().sum()}")
print(f"- metadata creator unique accounts: {len(metadata_creators)}")
print(f"- creators also commenters: {len(set(metadata_creators) & set(commenters))}")
print(f"- creators without comments: {len(set(metadata_creators) - set(commenters))}")
print(f"- creators also HCC: {len(set(metadata_creators) & hcc_set)}")


Video creator audit:
- metadata creator raw rows: 55
- metadata creator unique accounts: 43
- creators also commenters: 40
- creators without comments: 3
- creators also HCC: 0


In [6]:
# 9. Load dan apply verified registry
registry_verified = registry.loc[registry["verification_status_norm"].eq("verified") & registry["username_norm"].ne("")].copy()
registry_candidate = registry.loc[registry["verification_status_norm"].eq("candidate") & registry["username_norm"].ne("")].copy()
registry_rejected = registry.loc[registry["verification_status_norm"].eq("rejected") & registry["username_norm"].ne("")].copy()

registry_verified_map = registry_verified.drop_duplicates("username_norm", keep="last").set_index("username_norm").to_dict("index")
allowed_verified_subtypes = {
    "Official Brand Account",
    "Influencer",
    "Owner / Business Representative",
    "Expert / Professional",
    "Other Individual Source",
    "Video Creator",
}

def registry_value(username_norm, field, default=""):
    data = registry_verified_map.get(username_norm, {})
    value = data.get(field, default)
    return str(value).strip() if pd.notna(value) else default

def final_individual_subtype(row):
    if row["actor_type_primary"] != "Individual Actor":
        return ""
    subtype = registry_value(row["username_norm"], "individual_subtype", "")
    if subtype in allowed_verified_subtypes:
        return subtype
    if row["is_video_creator"]:
        return "Video Creator"
    return "Other Individual Source"

def final_individual_source(row):
    sources = []
    if row["is_video_creator"]:
        sources.append("video_metadata_clean.csv:creator")
    source = registry_value(row["username_norm"], "identification_source", "")
    if row["is_verified_registry_actor"]:
        sources.append(source or "individual_actor_registry.csv")
    return "; ".join(sources)

def final_individual_verification(row):
    if row["actor_type_primary"] != "Individual Actor":
        return ""
    if row["is_video_creator"] and not row["is_verified_registry_actor"]:
        return "Metadata-confirmed creator"
    if row["is_video_creator"] and row["is_verified_registry_actor"]:
        return "Metadata-confirmed creator; Verified registry"
    if row["is_verified_registry_actor"]:
        return "Verified registry"
    return ""

actors["individual_subtype"] = actors.apply(final_individual_subtype, axis=1)
actors["individual_identification_source"] = actors.apply(final_individual_source, axis=1)
actors["individual_verification_level"] = actors.apply(final_individual_verification, axis=1)
actors["registry_display_name"] = actors["username_norm"].map(lambda x: registry_value(x, "display_name", ""))
actors["registry_associated_brand"] = actors["username_norm"].map(lambda x: registry_value(x, "associated_brand", ""))
actors["display_name"] = np.where(actors["registry_display_name"].ne(""), actors["registry_display_name"], actors["username"])

individual_actor_candidates = pd.concat(
    [
        video_metadata.loc[video_metadata["creator_norm"].ne(""), ["creator", "creator_norm", "product_category", "metadata_brand"]]
        .drop_duplicates()
        .assign(candidate_source="video_metadata_clean.csv:creator", candidate_status="Metadata-confirmed creator"),
        registry.loc[registry["username_norm"].ne(""), ["username", "username_norm", "individual_subtype", "associated_brand"]]
        .drop_duplicates()
        .rename(columns={"username": "creator", "associated_brand": "metadata_brand"})
        .assign(product_category="", candidate_source="individual_actor_registry.csv", candidate_status=registry["verification_status"].values[: len(registry.loc[registry["username_norm"].ne("")])]),
    ],
    ignore_index=True,
    sort=False,
)
if individual_actor_candidates.empty:
    individual_actor_candidates = pd.DataFrame(
        columns=["creator", "creator_norm", "product_category", "metadata_brand", "candidate_source", "candidate_status"]
    )

print("Registry audit:")
print(f"- verified registry actors: {len(verified_registry)}")
print(f"- candidate registry actors: {len(registry_candidate)}")
print(f"- rejected registry actors ignored: {len(registry_rejected)}")
print(f"- verified registry non-creator actors: {len(set(verified_registry) - set(metadata_creators))}")


Registry audit:
- verified registry actors: 0
- candidate registry actors: 0
- rejected registry actors ignored: 0
- verified registry non-creator actors: 0


In [7]:
# 10. Actor type classification
if actors["username_norm"].duplicated().any():
    duplicates = actors.loc[actors["username_norm"].duplicated(), "username_norm"].tolist()
    raise AssertionError(f"Duplicate actor rows found: {duplicates[:10]}")
if set(actors["actor_type_primary"].unique()) - set(VALID_ACTOR_TYPES):
    raise AssertionError("Actor type outside the three-category framework found.")
missing_creators = set(metadata_creators) - set(actors.loc[actors["actor_type_primary"].eq("Individual Actor"), "username_norm"])
if missing_creators:
    raise AssertionError(f"Metadata creators not classified as Individual Actor: {sorted(missing_creators)[:10]}")
hcc_mass = actors.loc[actors["is_hcc_member"] & actors["actor_type_primary"].eq("Mass Actor")]
if not hcc_mass.empty:
    raise AssertionError("At least one HCC member was classified as Mass Actor.")

actor_type_counts = actors["actor_type_primary"].value_counts().reindex(VALID_ACTOR_TYPES, fill_value=0)
print("Actor type counts:")
print(actor_type_counts.to_string())


Actor type counts:
actor_type_primary
Individual Actor       43
Community Actor       218
Mass Actor          26166


In [8]:
# 11. Community Actor attributes
hcc_nodes_numeric = hcc_nodes.copy()
for col in ["degree", "weighted_degree", "betweenness"]:
    hcc_nodes_numeric[col] = to_numeric(hcc_nodes_numeric[col], fill_value=0)
hcc_nodes_numeric["community"] = hcc_nodes_numeric["community"].astype(str)
hcc_sentiment["hcc_id"] = hcc_sentiment["hcc_id"].astype(str)

hcc_account_membership = actors.loc[actors["is_hcc_member"], ["username_norm", "actor_type_primary", "community"]].copy()
hcc_account_membership["community"] = hcc_account_membership["community"].astype(str)
hcc_member_counts = (
    hcc_account_membership.groupby("community")
    .agg(
        total_hcc_members=("username_norm", "nunique"),
        community_actor_accounts=("actor_type_primary", lambda s: int((s == "Community Actor").sum())),
        individual_hcc_accounts=("actor_type_primary", lambda s: int((s == "Individual Actor").sum())),
    )
    .reset_index()
    .rename(columns={"community": "hcc_id"})
)

hcc_centrality = (
    hcc_nodes_numeric.groupby("community")
    .agg(
        hcc_mean_degree=("degree", "mean"),
        hcc_mean_weighted_degree=("weighted_degree", "mean"),
        hcc_mean_betweenness=("betweenness", "mean"),
    )
    .reset_index()
    .rename(columns={"community": "hcc_id"})
)
hcc_reference = (
    hcc_sentiment.merge(hcc_member_counts, on="hcc_id", how="left")
    .merge(hcc_centrality, on="hcc_id", how="left")
)
for col in ["total_hcc_members", "community_actor_accounts", "individual_hcc_accounts"]:
    hcc_reference[col] = to_numeric(hcc_reference[col], fill_value=0).astype(int)

print("Community audit:")
print(f"- HCC communities: {hcc_reference['hcc_id'].nunique()}")
print(f"- HCC accounts found in dataset/account universe: {actors['is_hcc_member'].sum()}")


Community audit:
- HCC communities: 42
- HCC accounts found in dataset/account universe: 218


In [9]:
# 12. Mass Actor identification
actor_type_map = actors.set_index("username_norm")["actor_type_primary"].to_dict()
public_name_map = actors.set_index("username_norm")["public_username"].to_dict()
anonymous_id_map = actors.set_index("username_norm")["actor_id_anonymous"].to_dict()
community_map = actors.set_index("username_norm")["community"].fillna("").astype(str).to_dict()

comments = comment_sentiment.copy()
comments = comments.merge(
    dataset[["comment_id_norm", "comment_id_float_key", "parent_user_norm", "parent_comment_id_norm", "video_creator_norm", "source_file"]]
    .drop_duplicates("comment_id_norm"),
    on="comment_id_norm",
    how="left",
)
comments["actor_type_primary"] = comments["username_norm"].map(actor_type_map).fillna("Mass Actor")
comments["public_username"] = comments["username_norm"].map(public_name_map).fillna(comments["username_norm"].map(public_mass_id))
comments["actor_id_anonymous"] = comments["username_norm"].map(anonymous_id_map).fillna(comments["username_norm"].map(public_mass_id))
comments["hcc_id"] = comments["username_norm"].map(community_map).fillna("")
comments["is_mass_actor_comment"] = comments["actor_type_primary"].eq("Mass Actor")
comments["is_community_actor_comment"] = comments["actor_type_primary"].eq("Community Actor")
comments["is_individual_actor_comment"] = comments["actor_type_primary"].eq("Individual Actor")

mass_actors = actors.loc[actors["actor_type_primary"].eq("Mass Actor"), "username_norm"].tolist()
print("Mass audit:")
print(f"- Mass Actor accounts: {len(mass_actors)}")
print(f"- LCN Non-HCC Mass Actor accounts: {(actors['mass_position_secondary'] == 'LCN Non-HCC').sum()}")
print(f"- Outside LCN Mass Actor accounts: {(actors['mass_position_secondary'] == 'Outside LCN').sum()}")


Mass audit:
- Mass Actor accounts: 26166
- LCN Non-HCC Mass Actor accounts: 506
- Outside LCN Mass Actor accounts: 25660


In [10]:
# 13. Parent reply validation
comment_lookup_cols = [
    "comment_id_norm",
    "comment_id_float_key",
    "username_norm",
    "public_username",
    "video_id_norm",
    "timestamp_dt",
    "actor_type_primary",
    "hcc_id",
    "sentiment_label",
]
comment_lookup_exact = comments[comment_lookup_cols].drop_duplicates("comment_id_norm").set_index("comment_id_norm")
comment_lookup_float = (
    comments.loc[comments["comment_id_float_key"].ne(""), comment_lookup_cols]
    .drop_duplicates("comment_id_float_key")
    .set_index("comment_id_float_key")
)

def lookup_parent(parent_id_norm: str):
    if parent_id_norm in comment_lookup_exact.index:
        return comment_lookup_exact.loc[parent_id_norm]
    if parent_id_norm in comment_lookup_float.index:
        return comment_lookup_float.loc[parent_id_norm]
    return None

reply_rows = comments.loc[comments["parent_comment_id_norm"].fillna("").ne("")].copy()
reply_audit_records = []
verified_direct_records = []

for row in reply_rows.itertuples(index=False):
    parent_id = getattr(row, "parent_comment_id_norm")
    parent = lookup_parent(parent_id)
    child_type = getattr(row, "actor_type_primary")
    child_user = getattr(row, "username_norm")
    child_public = getattr(row, "public_username")
    child_hcc = str(getattr(row, "hcc_id") or "")
    parent_user_reported_norm = getattr(row, "parent_user_norm") or ""
    parent_user_reported_public = public_name_map.get(parent_user_reported_norm, public_mass_id(parent_user_reported_norm)) if parent_user_reported_norm else ""
    status = "Invalid Parent Reference"
    parent_public = ""
    parent_video = ""
    associated_hcc = ""
    notes = ""
    parent_type = ""
    parent_user_norm = ""
    minutes_after_hcc = np.nan

    if parent is None:
        status = "Missing Parent Comment"
        associated_hcc = community_map.get(parent_user_reported_norm, "")
        notes = "Parent comment ID could not be matched after exact and float-rounded ID normalization."
    else:
        parent_user_norm = str(parent["username_norm"])
        parent_public = str(parent["public_username"])
        parent_video = str(parent["video_id_norm"])
        parent_type = str(parent["actor_type_primary"])
        parent_hcc = str(parent["hcc_id"] or "")
        same_video = str(getattr(row, "video_id_norm")) == parent_video
        parent_user_match = (not parent_user_reported_norm) or parent_user_reported_norm == parent_user_norm
        if not same_video:
            status = "Cross-Video Parent Mismatch"
            notes = "Parent comment matched but belongs to a different video."
        elif not parent_user_match:
            status = "Parent User Mismatch"
            associated_hcc = parent_hcc if parent_hcc else child_hcc
            notes = "Parent comment username does not match parent_user field."
        else:
            pair_is_mass_to_hcc = child_type == "Mass Actor" and parent_type == "Community Actor"
            pair_is_hcc_to_mass = child_type == "Community Actor" and parent_type == "Mass Actor"
            if pair_is_mass_to_hcc or pair_is_hcc_to_mass:
                status = "Verified Direct Reply"
                associated_hcc = parent_hcc if pair_is_mass_to_hcc else child_hcc
                notes = "Parent lookup, video, parent_user, and actor categories validated."
                if pd.notna(getattr(row, "timestamp_dt")) and pd.notna(parent["timestamp_dt"]):
                    delta = (getattr(row, "timestamp_dt") - parent["timestamp_dt"]).total_seconds() / 60
                    minutes_after_hcc = delta if pair_is_mass_to_hcc else np.nan
                mass_user_norm = child_user if pair_is_mass_to_hcc else parent_user_norm
                mass_public = child_public if pair_is_mass_to_hcc else parent_public
                mass_comment_id = getattr(row, "comment_id_norm") if pair_is_mass_to_hcc else parent.name
                mass_video = getattr(row, "video_id_norm") if pair_is_mass_to_hcc else parent_video
                mass_timestamp = getattr(row, "timestamp_dt") if pair_is_mass_to_hcc else parent["timestamp_dt"]
                mass_sentiment = getattr(row, "sentiment_label") if pair_is_mass_to_hcc else parent["sentiment_label"]
                verified_direct_records.append(
                    {
                        "mass_username_norm": mass_user_norm,
                        "mass_username": mass_public,
                        "mass_actor_id_anonymous": anonymous_id_map.get(mass_user_norm, public_mass_id(mass_user_norm)),
                        "hcc_id": associated_hcc,
                        "comment_id": mass_comment_id,
                        "video_id": mass_video,
                        "timestamp": mass_timestamp,
                        "sentiment_label": mass_sentiment,
                        "association_type": "Direct Interaction",
                        "minutes_after_hcc": minutes_after_hcc,
                        "is_verified_direct_reply": True,
                        "direction": "Mass replies to HCC" if pair_is_mass_to_hcc else "HCC replies to Mass",
                    }
                )
            else:
                status = "Invalid Parent Reference"
                associated_hcc = parent_hcc if parent_type == "Community Actor" else child_hcc
                notes = "Reply exists, but actor categories are not Mass-Community or Community-Mass."

    reply_audit_records.append(
        {
            "child_comment_id": getattr(row, "comment_id_norm"),
            "child_username": child_public,
            "child_actor_type": child_type,
            "parent_comment_id": parent_id,
            "parent_user_reported": parent_user_reported_public,
            "parent_username_lookup": parent_public,
            "child_video_id": getattr(row, "video_id_norm"),
            "parent_video_id": parent_video,
            "validation_status": status,
            "associated_hcc": associated_hcc,
            "notes": notes,
        }
    )

reply_validation_audit = pd.DataFrame(reply_audit_records)
if reply_validation_audit.empty:
    reply_validation_audit = pd.DataFrame(
        columns=[
            "child_comment_id",
            "child_username",
            "child_actor_type",
            "parent_comment_id",
            "parent_user_reported",
            "parent_username_lookup",
            "child_video_id",
            "parent_video_id",
            "validation_status",
            "associated_hcc",
            "notes",
        ]
    )
verified_direct = pd.DataFrame(verified_direct_records)

reply_status_counts = reply_validation_audit["validation_status"].value_counts().to_dict()
print("Reply validation status:")
for key, value in reply_status_counts.items():
    print(f"- {key}: {value}")


Reply validation status:
- Invalid Parent Reference: 8236
- Parent User Mismatch: 982
- Missing Parent Comment: 65
- Verified Direct Reply: 63


In [11]:
# 14. Shared-video context mapping
shared_records = []
for video_id, group in comments.groupby("video_id_norm", dropna=False):
    if not video_id:
        continue
    mass_users = sorted(set(group.loc[group["actor_type_primary"].eq("Mass Actor"), "username_norm"]) - {""})
    hcc_ids = sorted(set(group.loc[group["actor_type_primary"].eq("Community Actor"), "hcc_id"].astype(str)) - {""})
    if not mass_users or not hcc_ids:
        continue
    for mass_user in mass_users:
        for hcc_id in hcc_ids:
            shared_records.append(
                {
                    "mass_username_norm": mass_user,
                    "hcc_id": hcc_id,
                    "video_id": video_id,
                    "shared_video_context_count": 1,
                }
            )

shared_video_context = pd.DataFrame(shared_records)
if shared_video_context.empty:
    shared_video_agg = pd.DataFrame(columns=["mass_username_norm", "hcc_id", "shared_video_context_count"])
else:
    shared_video_agg = (
        shared_video_context.groupby(["mass_username_norm", "hcc_id"], as_index=False)
        .agg(shared_video_context_count=("video_id", "nunique"))
    )

print("Shared-video context records:")
print(f"- many-to-many mass-HCC video contexts: {len(shared_video_context)}")
print(f"- aggregated mass-HCC context pairs: {len(shared_video_agg)}")


Shared-video context records:


- many-to-many mass-HCC video contexts: 108674
- aggregated mass-HCC context pairs: 107141


In [12]:
# 15. Temporal association mapping
window_ns = MASS_TEMPORAL_WINDOW_MINUTES * 60 * 1_000_000_000
temporal_records = []
temporal_comment_rows = []

for video_id, group in comments.loc[comments["timestamp_dt"].notna()].groupby("video_id_norm", dropna=False):
    if not video_id:
        continue
    mass_group = group.loc[group["actor_type_primary"].eq("Mass Actor")].copy()
    hcc_group = group.loc[group["actor_type_primary"].eq("Community Actor") & group["hcc_id"].astype(str).ne("")].copy()
    if mass_group.empty or hcc_group.empty:
        continue
    for hcc_id, hcc_comments in hcc_group.groupby("hcc_id"):
        hcc_comments = hcc_comments.sort_values("timestamp_dt")
        hcc_times = hcc_comments["timestamp_dt"].astype("int64").to_numpy()
        if len(hcc_times) == 0:
            continue
        for mass_row in mass_group.itertuples(index=False):
            mass_time = pd.Timestamp(getattr(mass_row, "timestamp_dt")).value
            lower = mass_time - window_ns
            lo = np.searchsorted(hcc_times, lower, side="left")
            hi = np.searchsorted(hcc_times, mass_time, side="right")
            pair_count = int(hi - lo)
            if pair_count <= 0:
                continue
            matched_times = hcc_times[lo:hi]
            minutes = (mass_time - matched_times) / 1_000_000_000 / 60
            nearest = float(np.min(minutes))
            temporal_records.append(
                {
                    "mass_username_norm": getattr(mass_row, "username_norm"),
                    "hcc_id": str(hcc_id),
                    "comment_id": getattr(mass_row, "comment_id_norm"),
                    "video_id": video_id,
                    "temporal_hcc_comment_pair_count": pair_count,
                    "nearest_prior_hcc_comment_minutes": nearest,
                    "min_minutes_after_hcc": float(np.min(minutes)),
                    "median_minutes_after_hcc": float(np.median(minutes)),
                }
            )
            temporal_comment_rows.append(
                {
                    "comment_id": getattr(mass_row, "comment_id_norm"),
                    "mass_username_norm": getattr(mass_row, "username_norm"),
                    "mass_username": getattr(mass_row, "public_username"),
                    "video_id": video_id,
                    "timestamp": getattr(mass_row, "timestamp_dt"),
                    "sentiment_label": getattr(mass_row, "sentiment_label"),
                    "hcc_id": str(hcc_id),
                    "association_type": "Temporal Association",
                    "minutes_after_hcc": nearest,
                    "is_verified_direct_reply": False,
                }
            )

temporal_raw = pd.DataFrame(temporal_records)
if temporal_raw.empty:
    temporal_agg = pd.DataFrame(
        columns=[
            "mass_username_norm",
            "hcc_id",
            "temporal_mass_comment_count",
            "temporal_hcc_comment_pair_count",
            "n_unique_temporal_mass_comments",
            "n_temporal_comment_pairs",
            "min_minutes_after_hcc",
            "median_minutes_after_hcc",
            "nearest_prior_hcc_comment_minutes",
        ]
    )
else:
    temporal_agg = (
        temporal_raw.groupby(["mass_username_norm", "hcc_id"], as_index=False)
        .agg(
            temporal_mass_comment_count=("comment_id", "nunique"),
            temporal_hcc_comment_pair_count=("temporal_hcc_comment_pair_count", "sum"),
            n_unique_temporal_mass_comments=("comment_id", "nunique"),
            n_temporal_comment_pairs=("temporal_hcc_comment_pair_count", "sum"),
            min_minutes_after_hcc=("min_minutes_after_hcc", "min"),
            median_minutes_after_hcc=("median_minutes_after_hcc", "median"),
            nearest_prior_hcc_comment_minutes=("nearest_prior_hcc_comment_minutes", "min"),
        )
    )

temporal_comment_alignment_seed = pd.DataFrame(temporal_comment_rows)
print("Temporal association audit:")
print(f"- temporal mass comment-HCC rows: {len(temporal_raw)}")
print(f"- unique temporal mass comments: {temporal_raw['comment_id'].nunique() if not temporal_raw.empty else 0}")
print(f"- temporal HCC comment pairs: {int(temporal_raw['temporal_hcc_comment_pair_count'].sum()) if not temporal_raw.empty else 0}")


Temporal association audit:
- temporal mass comment-HCC rows: 2825
- unique temporal mass comments: 2409
- temporal HCC comment pairs: 5254


In [13]:
# 16. Mass-HCC many-to-many association
direct_agg = pd.DataFrame(
    columns=[
        "mass_username_norm",
        "hcc_id",
        "verified_direct_reply_count",
        "mass_replies_to_hcc_verified_count",
        "hcc_replies_to_mass_verified_count",
    ]
)
if not verified_direct.empty:
    verified_direct["mass_replies_to_hcc_verified_count"] = (verified_direct["direction"] == "Mass replies to HCC").astype(int)
    verified_direct["hcc_replies_to_mass_verified_count"] = (verified_direct["direction"] == "HCC replies to Mass").astype(int)
    direct_agg = (
        verified_direct.groupby(["mass_username_norm", "hcc_id"], as_index=False)
        .agg(
            verified_direct_reply_count=("comment_id", "count"),
            mass_replies_to_hcc_verified_count=("mass_replies_to_hcc_verified_count", "sum"),
            hcc_replies_to_mass_verified_count=("hcc_replies_to_mass_verified_count", "sum"),
        )
    )

unverified_status_columns = {
    "Missing Parent Comment": "parent_comment_missing_count",
    "Parent User Mismatch": "parent_user_mismatch_count",
    "Cross-Video Parent Mismatch": "cross_video_parent_mismatch_count",
}
unverified_reply_refs = reply_validation_audit.loc[reply_validation_audit["validation_status"].ne("Verified Direct Reply")].copy()
if not unverified_reply_refs.empty:
    parent_public_to_norm = {v: k for k, v in public_name_map.items()}
    unverified_reply_refs["mass_username_norm"] = unverified_reply_refs["child_username"].map(parent_public_to_norm)
    unverified_reply_refs["mass_username_norm"] = unverified_reply_refs["mass_username_norm"].fillna("")
    unverified_reply_refs = unverified_reply_refs.loc[
        unverified_reply_refs["associated_hcc"].astype(str).ne("") & unverified_reply_refs["mass_username_norm"].ne("")
    ]
    for status_name, col_name in unverified_status_columns.items():
        unverified_reply_refs[col_name] = unverified_reply_refs["validation_status"].eq(status_name).astype(int)
    unverified_agg = (
        unverified_reply_refs.groupby(["mass_username_norm", "associated_hcc"], as_index=False)
        .agg(
            unverified_reply_reference_count=("validation_status", "count"),
            parent_comment_missing_count=("parent_comment_missing_count", "sum"),
            parent_user_mismatch_count=("parent_user_mismatch_count", "sum"),
            cross_video_parent_mismatch_count=("cross_video_parent_mismatch_count", "sum"),
        )
        .rename(columns={"associated_hcc": "hcc_id"})
    )
else:
    unverified_agg = pd.DataFrame(
        columns=[
            "mass_username_norm",
            "hcc_id",
            "unverified_reply_reference_count",
            "parent_comment_missing_count",
            "parent_user_mismatch_count",
            "cross_video_parent_mismatch_count",
        ]
    )

association_keys = pd.concat(
    [
        shared_video_agg[["mass_username_norm", "hcc_id"]],
        temporal_agg[["mass_username_norm", "hcc_id"]],
        direct_agg[["mass_username_norm", "hcc_id"]],
        unverified_agg[["mass_username_norm", "hcc_id"]],
    ],
    ignore_index=True,
).drop_duplicates()

mass_hcc_association = association_keys.copy()
for frame in [shared_video_agg, temporal_agg, direct_agg, unverified_agg]:
    mass_hcc_association = mass_hcc_association.merge(frame, on=["mass_username_norm", "hcc_id"], how="left")

count_cols = [
    "shared_video_context_count",
    "temporal_mass_comment_count",
    "temporal_hcc_comment_pair_count",
    "n_unique_temporal_mass_comments",
    "n_temporal_comment_pairs",
    "verified_direct_reply_count",
    "mass_replies_to_hcc_verified_count",
    "hcc_replies_to_mass_verified_count",
    "unverified_reply_reference_count",
    "parent_comment_missing_count",
    "parent_user_mismatch_count",
    "cross_video_parent_mismatch_count",
]
for col in count_cols:
    if col not in mass_hcc_association.columns:
        mass_hcc_association[col] = 0
    mass_hcc_association[col] = to_numeric(mass_hcc_association[col], fill_value=0).astype(int)

mass_hcc_association["verified_direct_interaction_count"] = mass_hcc_association["verified_direct_reply_count"]
mass_hcc_association["strong_association_score"] = (
    3 * mass_hcc_association["verified_direct_interaction_count"]
    + 2 * mass_hcc_association["temporal_mass_comment_count"]
)
mass_hcc_association["contextual_association_score"] = (
    mass_hcc_association["strong_association_score"] + mass_hcc_association["shared_video_context_count"]
)
mass_hcc_association["mass_actor_id_anonymous"] = mass_hcc_association["mass_username_norm"].map(anonymous_id_map)
mass_hcc_association["mass_username"] = mass_hcc_association["mass_username_norm"].map(public_name_map)

print("Mass-HCC association pairs:")
print(f"- many-to-many pairs: {len(mass_hcc_association)}")
print(f"- pairs with verified direct interaction: {(mass_hcc_association['verified_direct_interaction_count'] > 0).sum()}")
print(f"- pairs with temporal association: {(mass_hcc_association['temporal_mass_comment_count'] > 0).sum()}")
print(f"- pairs with shared-video context: {(mass_hcc_association['shared_video_context_count'] > 0).sum()}")


Mass-HCC association pairs:
- many-to-many pairs: 112384
- pairs with verified direct interaction: 28
- pairs with temporal association: 2689
- pairs with shared-video context: 107141


In [14]:
# 17. Unique and ambiguous HCC association
mass_assoc_records = []
assoc_by_mass = {key: grp.copy() for key, grp in mass_hcc_association.groupby("mass_username_norm")} if not mass_hcc_association.empty else {}

for mass_user in mass_actors:
    grp = assoc_by_mass.get(mass_user, pd.DataFrame())
    associated_hcc_ids = []
    status = "No HCC Association"
    primary = ""
    basis = ""
    ambiguous = False
    if not grp.empty:
        grp = grp.copy()
        associated_hcc_ids = sorted(set(grp["hcc_id"].astype(str)) - {""})
        direct_max = int(grp["verified_direct_interaction_count"].max())
        temporal_max = int(grp["temporal_mass_comment_count"].max())
        shared_positive = grp.loc[grp["shared_video_context_count"].gt(0)]
        if direct_max > 0:
            winners = sorted(grp.loc[grp["verified_direct_interaction_count"].eq(direct_max), "hcc_id"].astype(str).unique())
            if len(winners) == 1:
                status = "Unique Direct Association"
                primary = winners[0]
                basis = "Verified direct reply count"
            else:
                status = "Multiple / Ambiguous HCC"
                basis = "Direct interaction tie"
                ambiguous = True
        elif temporal_max > 0:
            winners = sorted(grp.loc[grp["temporal_mass_comment_count"].eq(temporal_max), "hcc_id"].astype(str).unique())
            if len(winners) == 1:
                status = "Unique Temporal Association"
                primary = winners[0]
                basis = "Unique temporal mass comment count"
            else:
                status = "Multiple / Ambiguous HCC"
                basis = "Temporal association tie"
                ambiguous = True
        elif len(shared_positive) > 0:
            winners = sorted(shared_positive["hcc_id"].astype(str).unique())
            if len(winners) == 1:
                status = "Single Shared-Video Context"
                primary = winners[0]
                basis = "Single shared-video context only"
            else:
                status = "Multiple / Ambiguous HCC"
                basis = "Multiple shared-video contexts only"
                ambiguous = True
        else:
            status = "No HCC Association"
            basis = ""
    n_assoc = len(associated_hcc_ids)
    fractional = 1 / n_assoc if n_assoc else 0
    mass_assoc_records.append(
        {
            "username_norm": mass_user,
            "hcc_association_status": status,
            "primary_associated_hcc": primary,
            "associated_hcc_ids": ";".join(associated_hcc_ids),
            "n_associated_hcc": n_assoc,
            "primary_association_basis": basis,
            "is_ambiguous_hcc_association": ambiguous,
            "mass_fractional_context_weight_per_hcc": fractional,
        }
    )

mass_assoc_status = pd.DataFrame(mass_assoc_records)
mass_totals = mass_hcc_association.groupby("mass_username_norm").agg(
    shared_video_context_count=("shared_video_context_count", "sum"),
    verified_direct_reply_count=("verified_direct_reply_count", "sum"),
    verified_direct_interaction_count=("verified_direct_interaction_count", "sum"),
    mass_replies_to_hcc_verified_count=("mass_replies_to_hcc_verified_count", "sum"),
    hcc_replies_to_mass_verified_count=("hcc_replies_to_mass_verified_count", "sum"),
    unverified_reply_reference_count=("unverified_reply_reference_count", "sum"),
    parent_comment_missing_count=("parent_comment_missing_count", "sum"),
    parent_user_mismatch_count=("parent_user_mismatch_count", "sum"),
    cross_video_parent_mismatch_count=("cross_video_parent_mismatch_count", "sum"),
    temporal_mass_comment_count=("temporal_mass_comment_count", "sum"),
    temporal_hcc_comment_pair_count=("temporal_hcc_comment_pair_count", "sum"),
    n_unique_temporal_mass_comments=("n_unique_temporal_mass_comments", "sum"),
    n_temporal_comment_pairs=("n_temporal_comment_pairs", "sum"),
    strong_association_score=("strong_association_score", "sum"),
    contextual_association_score=("contextual_association_score", "sum"),
).reset_index().rename(columns={"mass_username_norm": "username_norm"}) if not mass_hcc_association.empty else pd.DataFrame(columns=["username_norm"])

mass_exposure = actors.loc[actors["actor_type_primary"].eq("Mass Actor")].copy()
mass_exposure = mass_exposure.merge(mass_assoc_status, on="username_norm", how="left").merge(mass_totals, on="username_norm", how="left")
for col in [
    "shared_video_context_count",
    "verified_direct_reply_count",
    "verified_direct_interaction_count",
    "mass_replies_to_hcc_verified_count",
    "hcc_replies_to_mass_verified_count",
    "unverified_reply_reference_count",
    "parent_comment_missing_count",
    "parent_user_mismatch_count",
    "cross_video_parent_mismatch_count",
    "temporal_mass_comment_count",
    "temporal_hcc_comment_pair_count",
    "n_unique_temporal_mass_comments",
    "n_temporal_comment_pairs",
    "strong_association_score",
    "contextual_association_score",
    "n_associated_hcc",
]:
    if col not in mass_exposure.columns:
        mass_exposure[col] = 0
    mass_exposure[col] = to_numeric(mass_exposure[col], fill_value=0)
mass_exposure["hcc_association_status"] = mass_exposure["hcc_association_status"].fillna("No HCC Association")
mass_exposure["primary_associated_hcc"] = mass_exposure["primary_associated_hcc"].fillna("")
mass_exposure["associated_hcc_ids"] = mass_exposure["associated_hcc_ids"].fillna("")
mass_exposure["primary_association_basis"] = mass_exposure["primary_association_basis"].fillna("")
mass_exposure["is_ambiguous_hcc_association"] = mass_exposure["is_ambiguous_hcc_association"].fillna(False)
mass_exposure["exposure_level"] = np.select(
    [
        mass_exposure["verified_direct_interaction_count"].gt(0),
        mass_exposure["temporal_mass_comment_count"].gt(0),
        mass_exposure["shared_video_context_count"].gt(0),
    ],
    ["Direct Interaction", "Temporal Association", "Shared-Video Context Only"],
    default="No Observed Community Association",
)

mass_hcc_association = mass_hcc_association.merge(
    mass_assoc_status[["username_norm", "hcc_association_status", "primary_associated_hcc", "n_associated_hcc", "is_ambiguous_hcc_association"]]
    .rename(columns={"username_norm": "mass_username_norm"}),
    on="mass_username_norm",
    how="left",
)
mass_hcc_association["is_primary_association"] = (
    mass_hcc_association["hcc_id"].astype(str).eq(mass_hcc_association["primary_associated_hcc"].astype(str))
    & mass_hcc_association["primary_associated_hcc"].astype(str).ne("")
)
mass_hcc_association["fractional_context_weight"] = np.where(
    mass_hcc_association["n_associated_hcc"].gt(0),
    1 / mass_hcc_association["n_associated_hcc"],
    0,
)

print("HCC association status:")
print(mass_exposure["hcc_association_status"].value_counts().reindex(VALID_HCC_ASSOCIATION_STATUS, fill_value=0).to_string())


HCC association status:
hcc_association_status
Unique Direct Association         28
Unique Temporal Association     1871
Single Shared-Video Context     1396
Multiple / Ambiguous HCC       22871
No HCC Association                 0


In [15]:
# 18. Individual-HCC association
hcc_comments = comments.loc[comments["actor_type_primary"].eq("Community Actor") & comments["hcc_id"].astype(str).ne("")].copy()
individual_norm_set = set(actors.loc[actors["actor_type_primary"].eq("Individual Actor"), "username_norm"])
creator_hcc_comments = hcc_comments.loc[hcc_comments["video_creator_norm"].isin(individual_norm_set)].copy()
individual_hcc_records = []
for (creator_norm, hcc_id), grp in creator_hcc_comments.groupby(["video_creator_norm", "hcc_id"]):
    individual_row = actors.loc[actors["username_norm"].eq(creator_norm)].iloc[0]
    hcc_row = hcc_reference.loc[hcc_reference["hcc_id"].astype(str).eq(str(hcc_id))]
    individual_hcc_records.append(
        {
            "individual_username": individual_row["public_username"],
            "individual_subtype": individual_row["individual_subtype"],
            "hcc_id": str(hcc_id),
            "shared_video_count": grp["video_id_norm"].nunique(),
            "creator_video_count": int(individual_row["creator_video_count"]),
            "hcc_account_count": grp["username_norm"].nunique(),
            "hcc_comment_count": len(grp),
            "first_hcc_comment_timestamp": grp["timestamp_dt"].min(),
            "last_hcc_comment_timestamp": grp["timestamp_dt"].max(),
            "target_brand": join_unique(grp["product_brand"]),
            "hcc_goal_orientation": hcc_row["goal_orientation"].iloc[0] if not hcc_row.empty else "",
            "association_type": "Observed Creator-Community Context",
        }
    )
individual_hcc_association = pd.DataFrame(individual_hcc_records)
if individual_hcc_association.empty:
    individual_hcc_association = pd.DataFrame(
        columns=[
            "individual_username",
            "individual_subtype",
            "hcc_id",
            "shared_video_count",
            "creator_video_count",
            "hcc_account_count",
            "hcc_comment_count",
            "first_hcc_comment_timestamp",
            "last_hcc_comment_timestamp",
            "target_brand",
            "hcc_goal_orientation",
            "association_type",
        ]
    )
print("Individual-HCC associations:")
print(f"- rows: {len(individual_hcc_association)}")
print(f"- individual actors connected: {individual_hcc_association['individual_username'].nunique() if not individual_hcc_association.empty else 0}")


Individual-HCC associations:
- rows: 186
- individual actors connected: 43


In [16]:
# 19. Target mapping
def creator_target(row):
    if row["actor_type_primary"] != "Individual Actor":
        return pd.Series(
            {
                "creator_target_brand_primary": "",
                "creator_target_brand_label": "",
                "creator_target_scope": "",
                "creator_product_category_primary": "",
                "creator_target_confidence": "None",
                "creator_target_source": "",
            }
        )
    registry_brand = str(row.get("registry_associated_brand", "") or "").strip()
    if registry_brand:
        return pd.Series(
            {
                "creator_target_brand_primary": registry_brand,
                "creator_target_brand_label": registry_brand,
                "creator_target_scope": "Registry Override",
                "creator_product_category_primary": "",
                "creator_target_confidence": "High",
                "creator_target_source": "individual_actor_registry.csv:associated_brand",
            }
        )
    creator_norm = row["username_norm"]
    creator_meta = video_metadata.loc[video_metadata["creator_norm"].eq(creator_norm)].copy()
    if creator_meta.empty:
        return pd.Series(
            {
                "creator_target_brand_primary": "Unidentified",
                "creator_target_brand_label": "Unidentified",
                "creator_target_scope": "Unidentified Brand Context",
                "creator_product_category_primary": "",
                "creator_target_confidence": "None",
                "creator_target_source": "",
            }
        )
    brand_counts = creator_meta["metadata_brand"].value_counts()
    product_counts = creator_meta["product_category"].value_counts()
    primary_brand = brand_counts.index[0]
    scope = "Single-brand Context" if len(brand_counts) == 1 else "Cross-brand Context"
    confidence = "High" if len(brand_counts) == 1 else "Medium"
    return pd.Series(
        {
            "creator_target_brand_primary": primary_brand,
            "creator_target_brand_label": join_unique(creator_meta["metadata_brand"]),
            "creator_target_scope": scope,
            "creator_product_category_primary": product_counts.index[0],
            "creator_target_confidence": confidence,
            "creator_target_source": "video_metadata_clean.csv product_category",
        }
    )

actors = pd.concat([actors, actors.apply(creator_target, axis=1)], axis=1)

actor_comment_context = (
    comments.groupby("username_norm")
    .agg(
        observed_product_categories=("product_category", join_unique),
        observed_brands=("product_brand", join_unique),
        observed_video_count=("video_id_norm", "nunique"),
    )
    .reset_index()
)
actors = actors.merge(actor_comment_context, on="username_norm", how="left")
actors["observed_video_count"] = to_numeric(actors["observed_video_count"], fill_value=0).astype(int)

hcc_brand_map = hcc_reference.set_index("hcc_id")["brand_label_auto"].to_dict()
mass_primary_hcc_brand = mass_exposure.set_index("username_norm")["primary_associated_hcc"].map(hcc_brand_map).to_dict()

def final_target(row):
    if row["actor_type_primary"] == "Individual Actor":
        return pd.Series(
            {
                "target_brand_primary": row["creator_target_brand_primary"],
                "target_brand_label": row["creator_target_brand_label"],
                "target_scope": row["creator_target_scope"],
                "target_source": row["creator_target_source"],
                "target_confidence": row["creator_target_confidence"],
            }
        )
    if row["actor_type_primary"] == "Community Actor":
        brand = str(row.get("brand_label_auto", "") or row.get("primary_brand", "") or "").strip()
        return pd.Series(
            {
                "target_brand_primary": brand,
                "target_brand_label": brand,
                "target_scope": "HCC Brand Context" if brand else "Unidentified Brand Context",
                "target_source": "output/gephi/gephi_hcc_nodes.csv brand_label_auto",
                "target_confidence": str(row.get("brand_confidence", "") or "Low"),
            }
        )
    primary_hcc_brand = mass_primary_hcc_brand.get(row["username_norm"], "")
    if primary_hcc_brand:
        return pd.Series(
            {
                "target_brand_primary": primary_hcc_brand,
                "target_brand_label": primary_hcc_brand,
                "target_scope": "Associated HCC Brand Context",
                "target_source": "primary associated HCC brand context",
                "target_confidence": "Medium",
            }
        )
    brands = [b for b in str(row.get("observed_brands", "") or "").split(";") if b]
    if len(brands) == 1:
        scope = "Single-brand Comment Context"
        confidence = "Medium"
        primary = brands[0]
    elif len(brands) > 1:
        scope = "Cross-brand Comment Context"
        confidence = "Low"
        primary = brands[0]
    else:
        scope = "Unidentified Brand Context"
        confidence = "None"
        primary = "Unidentified"
    return pd.Series(
        {
            "target_brand_primary": primary,
            "target_brand_label": ";".join(brands) if brands else "Unidentified",
            "target_scope": scope,
            "target_source": "observed comment product_category",
            "target_confidence": confidence,
        }
    )

actors = pd.concat([actors, actors.apply(final_target, axis=1)], axis=1)


In [17]:
# 20. Account-level goals
def account_goal_row(row):
    n_comments = int(row["n_comments"])
    n_valid = int(row["n_valid_text_comments"])
    if n_comments == 0:
        return pd.Series(
            {
                "account_goal_orientation": "Not Observed from Comments",
                "account_goal_confidence": "None",
                "is_goal_evaluable": False,
                "goal_evaluation_reason": "No observed comments",
            }
        )
    if n_valid == 0:
        return pd.Series(
            {
                "account_goal_orientation": "Insufficient Text",
                "account_goal_confidence": "None",
                "is_goal_evaluable": False,
                "goal_evaluation_reason": "No valid text",
            }
        )
    if n_valid < ACCOUNT_GOAL_MIN_COMMENTS:
        return pd.Series(
            {
                "account_goal_orientation": "Insufficient Text",
                "account_goal_confidence": "None",
                "is_goal_evaluable": False,
                "goal_evaluation_reason": "Fewer than 3 valid comments",
            }
        )
    goal, confidence = sentiment_goal_from_distribution(
        row["positive_count"],
        row["neutral_count"],
        row["negative_count"],
        n_valid,
        min_comments=ACCOUNT_GOAL_MIN_COMMENTS,
        allow_insufficient=True,
    )
    return pd.Series(
        {
            "account_goal_orientation": goal,
            "account_goal_confidence": confidence,
            "is_goal_evaluable": True,
            "goal_evaluation_reason": "Sufficient comments",
        }
    )

actors = pd.concat([actors, actors.apply(account_goal_row, axis=1)], axis=1)
mass_exposure = mass_exposure.merge(
    actors[
        [
            "username_norm",
            "account_goal_orientation",
            "account_goal_confidence",
            "is_goal_evaluable",
            "goal_evaluation_reason",
            "target_brand_primary",
            "target_scope",
        ]
    ],
    on="username_norm",
    how="left",
)

print("Account-level goal coverage:")
print(actors.groupby("actor_type_primary")["is_goal_evaluable"].mean().reindex(VALID_ACTOR_TYPES).fillna(0).to_string())


Account-level goal coverage:
actor_type_primary
Individual Actor    0.860465
Community Actor     1.000000
Mass Actor          0.038905


In [18]:
# 21. Pooled actor-type goals
valid_sentiment_comments = comments.loc[comments["sentiment_label"].isin(["Positive", "Neutral", "Negative"])].copy()
pooled_records = []
for actor_type in VALID_ACTOR_TYPES:
    actor_subset = actors.loc[actors["actor_type_primary"].eq(actor_type)]
    comment_subset = valid_sentiment_comments.loc[valid_sentiment_comments["actor_type_primary"].eq(actor_type)]
    pooled_positive = int((comment_subset["sentiment_label"] == "Positive").sum())
    pooled_neutral = int((comment_subset["sentiment_label"] == "Neutral").sum())
    pooled_negative = int((comment_subset["sentiment_label"] == "Negative").sum())
    n_valid = int(len(comment_subset))
    n_comments = int(comments.loc[comments["actor_type_primary"].eq(actor_type)].shape[0])
    pooled_goal, pooled_conf = sentiment_goal_from_distribution(
        pooled_positive,
        pooled_neutral,
        pooled_negative,
        n_valid,
        min_comments=1,
        allow_insufficient=False,
    )
    if n_valid == 0:
        pooled_goal = "Not Observed from Comments"
        pooled_conf = "None"
    modal_goal = (
        actor_subset["account_goal_orientation"].mode().iloc[0]
        if not actor_subset["account_goal_orientation"].dropna().empty
        else "Not Observed from Comments"
    )
    pooled_records.append(
        {
            "actor_type_primary": actor_type,
            "n_accounts": int(len(actor_subset)),
            "n_accounts_with_comments": int(actor_subset["n_comments"].gt(0).sum()),
            "n_accounts_goal_evaluable": int(actor_subset["is_goal_evaluable"].sum()),
            "goal_evaluable_percentage": float(actor_subset["is_goal_evaluable"].mean() * 100) if len(actor_subset) else 0.0,
            "n_comments": n_comments,
            "n_valid_comments": n_valid,
            "pooled_positive_count": pooled_positive,
            "pooled_neutral_count": pooled_neutral,
            "pooled_negative_count": pooled_negative,
            "pooled_positive_ratio": pooled_positive / n_valid if n_valid else np.nan,
            "pooled_neutral_ratio": pooled_neutral / n_valid if n_valid else np.nan,
            "pooled_negative_ratio": pooled_negative / n_valid if n_valid else np.nan,
            "pooled_dominant_sentiment": dominant_sentiment_from_counts(pooled_positive, pooled_neutral, pooled_negative, n_valid),
            "pooled_goal_orientation": pooled_goal,
            "pooled_goal_confidence": pooled_conf,
            "modal_account_goal_orientation": modal_goal,
        }
    )
actor_type_goals_pooled = pd.DataFrame(pooled_records)
print("Pooled actor-type goals:")
print(actor_type_goals_pooled[["actor_type_primary", "n_valid_comments", "pooled_goal_orientation", "pooled_dominant_sentiment"]].to_string(index=False))


Pooled actor-type goals:
actor_type_primary  n_valid_comments pooled_goal_orientation pooled_dominant_sentiment
  Individual Actor              1384      Neutral Engagement                   Neutral
   Community Actor              1009             Mixed Goals                  Positive
        Mass Actor             31427      Neutral Engagement                   Neutral


In [19]:
# 22. Account-level sentiment alignment
hcc_sentiment_map = hcc_reference.set_index("hcc_id")["dominant_sentiment"].to_dict()

def account_alignment(row):
    status = row["hcc_association_status"]
    if status == "No HCC Association":
        return "No HCC Association"
    if status == "Multiple / Ambiguous HCC":
        return "Ambiguous HCC Association"
    if status == "Single Shared-Video Context":
        return "Context Only"
    if status not in {"Unique Direct Association", "Unique Temporal Association"}:
        return "No HCC Association"
    if not bool(row["is_goal_evaluable"]):
        return "Insufficient Text"
    hcc_dom = hcc_sentiment_map.get(str(row["primary_associated_hcc"]), "")
    mass_dom = str(row["dominant_sentiment"])
    if hcc_dom not in {"Positive", "Neutral", "Negative"} or mass_dom not in {"Positive", "Neutral", "Negative"}:
        return "Mixed / Inconclusive"
    return "Aligned" if hcc_dom == mass_dom else "Not aligned"

mass_exposure["hcc_dominant_sentiment"] = mass_exposure["primary_associated_hcc"].astype(str).map(hcc_sentiment_map)
mass_exposure["sentiment_alignment"] = mass_exposure.apply(account_alignment, axis=1)

account_alignment_counts = mass_exposure["sentiment_alignment"].value_counts().reindex(VALID_ALIGNMENT_VALUES, fill_value=0)
alignment_evaluable = int(account_alignment_counts["Aligned"] + account_alignment_counts["Not aligned"])
alignment_denominator = len(mass_exposure)
account_alignment_coverage = alignment_evaluable / alignment_denominator if alignment_denominator else 0
account_alignment_rate = (
    account_alignment_counts["Aligned"] / alignment_evaluable if alignment_evaluable else np.nan
)
account_alignment_reliability = "Very Low Coverage" if alignment_evaluable < 10 else coverage_label(account_alignment_coverage)

print("Account-level alignment:")
print(account_alignment_counts.to_string())
print(f"coverage={account_alignment_coverage:.4f}; rate={account_alignment_rate if not pd.isna(account_alignment_rate) else 'nan'}; reliability={account_alignment_reliability}")


Account-level alignment:
sentiment_alignment
Aligned                         28
Not aligned                     31
Mixed / Inconclusive            53
Insufficient Text             1787
Ambiguous HCC Association    22871
Context Only                  1396
No HCC Association               0
coverage=0.0023; rate=0.4745762711864407; reliability=Very Low Coverage


In [20]:
# 23. Comment-level sentiment alignment
direct_comment_seed = verified_direct.copy()
if direct_comment_seed.empty:
    direct_comment_seed = pd.DataFrame(
        columns=[
            "comment_id",
            "mass_username_norm",
            "mass_username",
            "video_id",
            "timestamp",
            "sentiment_label",
            "hcc_id",
            "association_type",
            "minutes_after_hcc",
            "is_verified_direct_reply",
        ]
    )
else:
    direct_comment_seed = direct_comment_seed[
        [
            "comment_id",
            "mass_username_norm",
            "mass_username",
            "video_id",
            "timestamp",
            "sentiment_label",
            "hcc_id",
            "association_type",
            "minutes_after_hcc",
            "is_verified_direct_reply",
        ]
    ].copy()

mass_comment_hcc_alignment = pd.concat([direct_comment_seed, temporal_comment_alignment_seed], ignore_index=True, sort=False)
if mass_comment_hcc_alignment.empty:
    mass_comment_hcc_alignment = pd.DataFrame(
        columns=[
            "comment_id",
            "mass_username",
            "video_id",
            "timestamp",
            "sentiment_label",
            "hcc_id",
            "association_type",
            "hcc_dominant_sentiment",
            "comment_sentiment_alignment",
            "minutes_after_hcc",
            "is_verified_direct_reply",
            "is_multi_hcc_comment_association",
        ]
    )
else:
    mass_comment_hcc_alignment["hcc_dominant_sentiment"] = mass_comment_hcc_alignment["hcc_id"].astype(str).map(hcc_sentiment_map)

    def comment_alignment(row):
        hcc_dom = row["hcc_dominant_sentiment"]
        sentiment = row["sentiment_label"]
        if sentiment not in {"Positive", "Neutral", "Negative"}:
            return "No Valid Sentiment"
        if hcc_dom not in {"Positive", "Neutral", "Negative"}:
            return "HCC Mixed / Inconclusive"
        return "Aligned" if sentiment == hcc_dom else "Not aligned"

    mass_comment_hcc_alignment["comment_sentiment_alignment"] = mass_comment_hcc_alignment.apply(comment_alignment, axis=1)
    hcc_per_comment = mass_comment_hcc_alignment.groupby("comment_id")["hcc_id"].nunique()
    mass_comment_hcc_alignment["is_multi_hcc_comment_association"] = mass_comment_hcc_alignment["comment_id"].map(hcc_per_comment).gt(1)
    mass_comment_hcc_alignment = mass_comment_hcc_alignment[
        [
            "comment_id",
            "mass_username",
            "video_id",
            "timestamp",
            "sentiment_label",
            "hcc_id",
            "association_type",
            "hcc_dominant_sentiment",
            "comment_sentiment_alignment",
            "minutes_after_hcc",
            "is_verified_direct_reply",
            "is_multi_hcc_comment_association",
        ]
    ]

comment_alignment_counts = mass_comment_hcc_alignment["comment_sentiment_alignment"].value_counts()
comment_alignment_denominator = len(mass_comment_hcc_alignment)
comment_alignment_evaluable = int(comment_alignment_counts.get("Aligned", 0) + comment_alignment_counts.get("Not aligned", 0))
comment_alignment_coverage = comment_alignment_evaluable / comment_alignment_denominator if comment_alignment_denominator else 0
print("Comment-level alignment:")
print(comment_alignment_counts.to_string())
print(f"coverage={comment_alignment_coverage:.4f}; rows={comment_alignment_denominator}")


Comment-level alignment:


comment_sentiment_alignment
HCC Mixed / Inconclusive    1568
Not aligned                  667
Aligned                      650
No Valid Sentiment             3
coverage=0.4560; rows=2888


In [21]:
# 24. Account-level three dimensions typology
account_typology = actors.copy()
account_typology = account_typology.merge(
    mass_exposure[
        [
            "username_norm",
            "exposure_level",
            "hcc_association_status",
            "primary_associated_hcc",
            "associated_hcc_ids",
            "n_associated_hcc",
            "primary_association_basis",
            "is_ambiguous_hcc_association",
            "shared_video_context_count",
            "verified_direct_interaction_count",
            "temporal_mass_comment_count",
            "temporal_hcc_comment_pair_count",
            "strong_association_score",
            "contextual_association_score",
            "sentiment_alignment",
            "hcc_dominant_sentiment",
        ]
    ],
    on="username_norm",
    how="left",
)
account_typology["exposure_level"] = account_typology["exposure_level"].fillna("")
account_typology["hcc_association_status"] = account_typology["hcc_association_status"].fillna("")
account_typology["sentiment_alignment"] = account_typology["sentiment_alignment"].fillna("")
account_typology["primary_associated_hcc"] = account_typology["primary_associated_hcc"].fillna("")
account_typology["associated_hcc_ids"] = account_typology["associated_hcc_ids"].fillna("")

account_output_columns = [
    "username",
    "actor_id_anonymous",
    "display_name",
    "actor_type_primary",
    "individual_subtype",
    "is_video_creator",
    "is_verified_registry_actor",
    "individual_identification_source",
    "individual_verification_level",
    "is_hcc_member",
    "individual_in_hcc",
    "community",
    "is_lcn_member",
    "mass_position_secondary",
    "n_comments",
    "n_valid_text_comments",
    "positive_count",
    "neutral_count",
    "negative_count",
    "positive_ratio",
    "neutral_ratio",
    "negative_ratio",
    "dominant_sentiment",
    "avg_sentiment_confidence",
    "target_brand_primary",
    "target_brand_label",
    "target_scope",
    "target_source",
    "target_confidence",
    "creator_target_brand_primary",
    "creator_target_brand_label",
    "creator_target_scope",
    "creator_product_category_primary",
    "creator_video_count",
    "creator_target_confidence",
    "creator_target_source",
    "account_goal_orientation",
    "account_goal_confidence",
    "is_goal_evaluable",
    "goal_evaluation_reason",
    "exposure_level",
    "hcc_association_status",
    "primary_associated_hcc",
    "associated_hcc_ids",
    "n_associated_hcc",
    "primary_association_basis",
    "is_ambiguous_hcc_association",
    "shared_video_context_count",
    "verified_direct_interaction_count",
    "temporal_mass_comment_count",
    "temporal_hcc_comment_pair_count",
    "strong_association_score",
    "contextual_association_score",
    "sentiment_alignment",
    "hcc_dominant_sentiment",
]
three_dimensions_typology_account = account_typology.copy()
three_dimensions_typology_account["username"] = three_dimensions_typology_account["public_username"]
three_dimensions_typology_account = three_dimensions_typology_account[account_output_columns].copy()


In [22]:
# 25. HCC-level three dimensions typology
unique_primary = mass_exposure.loc[mass_exposure["primary_associated_hcc"].astype(str).ne("")].copy()
hcc_unique_counts = (
    unique_primary.groupby("primary_associated_hcc")
    .agg(n_mass_unique_associated=("username_norm", "nunique"))
    .reset_index()
    .rename(columns={"primary_associated_hcc": "hcc_id"})
)
hcc_fractional = (
    mass_hcc_association.groupby("hcc_id")
    .agg(
        mass_fractional_context_weight=("fractional_context_weight", "sum"),
        n_mass_many_to_many_rows=("mass_username_norm", "nunique"),
        shared_video_context_count=("shared_video_context_count", "sum"),
        verified_direct_interaction_count=("verified_direct_interaction_count", "sum"),
        temporal_mass_comment_count=("temporal_mass_comment_count", "sum"),
        temporal_pair_count=("temporal_hcc_comment_pair_count", "sum"),
    )
    .reset_index()
    if not mass_hcc_association.empty
    else pd.DataFrame(columns=["hcc_id"])
)
hcc_status_counts = (
    mass_exposure.loc[mass_exposure["primary_associated_hcc"].astype(str).ne("")]
    .groupby(["primary_associated_hcc", "hcc_association_status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={"primary_associated_hcc": "hcc_id"})
)
for status in VALID_HCC_ASSOCIATION_STATUS:
    if status not in hcc_status_counts.columns:
        hcc_status_counts[status] = 0

hcc_alignment_rows = []
for hcc_id, hcc_row in hcc_reference.set_index("hcc_id").iterrows():
    relevant = mass_exposure.loc[mass_exposure["primary_associated_hcc"].astype(str).eq(str(hcc_id))]
    counts = relevant["sentiment_alignment"].value_counts()
    total_relevant = len(relevant)
    n_aligned = int(counts.get("Aligned", 0))
    n_not = int(counts.get("Not aligned", 0))
    n_evaluable = n_aligned + n_not
    n_mixed = int(counts.get("Mixed / Inconclusive", 0))
    n_insufficient = int(counts.get("Insufficient Text", 0))
    n_ambiguous = int(counts.get("Ambiguous HCC Association", 0))
    cov = n_evaluable / total_relevant if total_relevant else 0
    rate = n_aligned / n_evaluable if n_evaluable else np.nan
    reliability = "Very Low Coverage" if n_evaluable < 10 else coverage_label(cov)
    hcc_alignment_rows.append(
        {
            "hcc_id": str(hcc_id),
            "total_relevant_mass_actors": total_relevant,
            "n_alignment_evaluable": n_evaluable,
            "n_aligned": n_aligned,
            "n_not_aligned": n_not,
            "n_mixed_or_inconclusive": n_mixed,
            "n_insufficient": n_insufficient,
            "n_ambiguous": n_ambiguous,
            "alignment_coverage": cov,
            "alignment_rate": rate,
            "alignment_reliability": reliability,
        }
    )
hcc_alignment_coverage = pd.DataFrame(hcc_alignment_rows)

three_dimensions_typology_hcc = (
    hcc_reference.merge(hcc_unique_counts, on="hcc_id", how="left")
    .merge(hcc_fractional, on="hcc_id", how="left")
    .merge(hcc_alignment_coverage, on="hcc_id", how="left")
    .merge(hcc_status_counts[["hcc_id"] + VALID_HCC_ASSOCIATION_STATUS], on="hcc_id", how="left")
)
fill_hcc_cols = [
    "n_mass_unique_associated",
    "mass_fractional_context_weight",
    "n_mass_many_to_many_rows",
    "shared_video_context_count",
    "verified_direct_interaction_count",
    "temporal_mass_comment_count",
    "temporal_pair_count",
    "total_relevant_mass_actors",
    "n_alignment_evaluable",
    "n_aligned",
    "n_not_aligned",
    "n_mixed_or_inconclusive",
    "n_insufficient",
    "n_ambiguous",
] + VALID_HCC_ASSOCIATION_STATUS
for col in fill_hcc_cols:
    if col in three_dimensions_typology_hcc.columns:
        three_dimensions_typology_hcc[col] = to_numeric(three_dimensions_typology_hcc[col], fill_value=0)

hcc_association_ambiguity_summary = three_dimensions_typology_hcc[
    [
        "hcc_id",
        "n_mass_unique_associated",
        "mass_fractional_context_weight",
        "n_mass_many_to_many_rows",
        "Unique Direct Association",
        "Unique Temporal Association",
        "Single Shared-Video Context",
        "Multiple / Ambiguous HCC",
        "shared_video_context_count",
        "verified_direct_interaction_count",
        "temporal_mass_comment_count",
        "temporal_pair_count",
    ]
].copy()


In [23]:
# 26. Summary tables
total_accounts = len(actors)
total_comments = int(actors["n_comments"].sum())
actor_type_summary_rows = []
alignment_by_type = {}
for actor_type in VALID_ACTOR_TYPES:
    subset = actors.loc[actors["actor_type_primary"].eq(actor_type)]
    comment_count = int(subset["n_comments"].sum())
    pooled_row = actor_type_goals_pooled.loc[actor_type_goals_pooled["actor_type_primary"].eq(actor_type)].iloc[0]
    if actor_type == "Mass Actor":
        align_counts = mass_exposure["sentiment_alignment"].value_counts()
        total_rel = len(mass_exposure)
        n_aligned = int(align_counts.get("Aligned", 0))
        n_not = int(align_counts.get("Not aligned", 0))
        n_eval = n_aligned + n_not
        n_mixed = int(align_counts.get("Mixed / Inconclusive", 0))
        n_insuf = int(align_counts.get("Insufficient Text", 0))
        n_ambig = int(align_counts.get("Ambiguous HCC Association", 0))
        cov = n_eval / total_rel if total_rel else 0
        rate = n_aligned / n_eval if n_eval else np.nan
        rel = "Very Low Coverage" if n_eval < 10 else coverage_label(cov)
    else:
        total_rel = 0
        n_eval = n_aligned = n_not = n_mixed = n_insuf = n_ambig = 0
        cov = np.nan
        rate = np.nan
        rel = "Not Applicable"
    actor_type_summary_rows.append(
        {
            "actor_type_primary": actor_type,
            "n_accounts": int(len(subset)),
            "n_comments": comment_count,
            "account_share": len(subset) / total_accounts if total_accounts else 0,
            "comment_share": comment_count / total_comments if total_comments else 0,
            "account_percentage": (len(subset) / total_accounts * 100) if total_accounts else 0,
            "comment_percentage": (comment_count / total_comments * 100) if total_comments else 0,
            "n_accounts_with_comments": int(subset["n_comments"].gt(0).sum()),
            "n_accounts_goal_evaluable": int(subset["is_goal_evaluable"].sum()),
            "goal_evaluable_percentage": float(subset["is_goal_evaluable"].mean() * 100) if len(subset) else 0,
            "modal_account_goal_orientation": pooled_row["modal_account_goal_orientation"],
            "pooled_goal_orientation": pooled_row["pooled_goal_orientation"],
            "pooled_dominant_sentiment": pooled_row["pooled_dominant_sentiment"],
            "total_relevant_mass_actors": total_rel,
            "n_alignment_evaluable": n_eval,
            "n_aligned": n_aligned,
            "n_not_aligned": n_not,
            "n_mixed_or_inconclusive": n_mixed,
            "n_insufficient": n_insuf,
            "n_ambiguous": n_ambig,
            "alignment_coverage": cov,
            "alignment_rate": rate,
            "alignment_reliability": rel,
        }
    )
actor_type_summary = pd.DataFrame(actor_type_summary_rows)

actor_type_by_target = (
    actors.groupby(["actor_type_primary", "target_scope", "target_brand_primary"], dropna=False)
    .agg(n_accounts=("username_norm", "nunique"), n_comments=("n_comments", "sum"))
    .reset_index()
    .sort_values(["actor_type_primary", "n_accounts"], ascending=[True, False])
)
actor_type_by_goals = (
    actors.groupby(["actor_type_primary", "account_goal_orientation"], dropna=False)
    .agg(n_accounts=("username_norm", "nunique"), n_comments=("n_comments", "sum"))
    .reset_index()
    .sort_values(["actor_type_primary", "n_accounts"], ascending=[True, False])
)

individual_actor_summary = actors.loc[actors["actor_type_primary"].eq("Individual Actor")].copy()
individual_actor_summary = individual_actor_summary[
    [
        "public_username",
        "display_name",
        "individual_subtype",
        "is_video_creator",
        "is_verified_registry_actor",
        "individual_identification_source",
        "individual_verification_level",
        "individual_in_hcc",
        "creator_video_count",
        "creator_target_brand_primary",
        "creator_target_brand_label",
        "creator_target_scope",
        "creator_product_category_primary",
        "creator_target_confidence",
        "creator_target_source",
        "n_comments",
        "n_valid_text_comments",
        "dominant_sentiment",
        "account_goal_orientation",
        "account_goal_confidence",
    ]
].rename(columns={"public_username": "username"})

community_actor_summary = three_dimensions_typology_hcc.copy()

actor_universe_audit = pd.DataFrame(
    [
        {
            "source_group": "Commenters",
            "n_raw_accounts": int(account_sentiment["username"].notna().sum()),
            "n_normalized_accounts": int(account_sentiment["username_norm"].ne("").sum()),
            "n_unique_accounts": len(commenters),
            "n_overlap_comment_creator": len(set(commenters) & set(metadata_creators)),
            "n_overlap_creator_hcc": len(set(metadata_creators) & hcc_set),
            "n_overlap_registry_creator": len(set(verified_registry) & set(metadata_creators)),
            "n_added_noncomment_creator": 0,
            "n_added_registry_noncomment_actor": 0,
        },
        {
            "source_group": "Video Creators",
            "n_raw_accounts": int(video_metadata["creator"].notna().sum()),
            "n_normalized_accounts": int(video_metadata["creator_norm"].ne("").sum()),
            "n_unique_accounts": len(metadata_creators),
            "n_overlap_comment_creator": len(set(commenters) & set(metadata_creators)),
            "n_overlap_creator_hcc": len(set(metadata_creators) & hcc_set),
            "n_overlap_registry_creator": len(set(verified_registry) & set(metadata_creators)),
            "n_added_noncomment_creator": len(set(metadata_creators) - set(commenters)),
            "n_added_registry_noncomment_actor": 0,
        },
        {
            "source_group": "Verified Registry",
            "n_raw_accounts": int(registry_verified["username"].ne("").sum()) if not registry_verified.empty else 0,
            "n_normalized_accounts": int(registry_verified["username_norm"].ne("").sum()) if not registry_verified.empty else 0,
            "n_unique_accounts": len(verified_registry),
            "n_overlap_comment_creator": len(set(verified_registry) & set(commenters) & set(metadata_creators)),
            "n_overlap_creator_hcc": len(set(verified_registry) & set(metadata_creators) & hcc_set),
            "n_overlap_registry_creator": len(set(verified_registry) & set(metadata_creators)),
            "n_added_noncomment_creator": 0,
            "n_added_registry_noncomment_actor": len(set(verified_registry) - set(commenters) - set(metadata_creators)),
        },
        {
            "source_group": "Final Actor Universe",
            "n_raw_accounts": len(commenters) + len(metadata_creators) + len(verified_registry),
            "n_normalized_accounts": len(commenters) + len(metadata_creators) + len(verified_registry),
            "n_unique_accounts": len(actor_universe_norm),
            "n_overlap_comment_creator": len(set(commenters) & set(metadata_creators)),
            "n_overlap_creator_hcc": len(set(metadata_creators) & hcc_set),
            "n_overlap_registry_creator": len(set(verified_registry) & set(metadata_creators)),
            "n_added_noncomment_creator": len(set(metadata_creators) - set(commenters)),
            "n_added_registry_noncomment_actor": len(set(verified_registry) - set(commenters) - set(metadata_creators)),
        },
    ]
)

method_thresholds = pd.DataFrame(
    [
        {"parameter": "ACCOUNT_GOAL_MIN_COMMENTS", "value": ACCOUNT_GOAL_MIN_COMMENTS, "interpretation": "Minimum valid text comments for account-level goals."},
        {"parameter": "MASS_TEMPORAL_WINDOW_MINUTES", "value": MASS_TEMPORAL_WINDOW_MINUTES, "interpretation": "Mass comment must appear after an HCC comment in this window."},
        {"parameter": "ANONYMIZE_PUBLIC_MASS_ACTORS", "value": ANONYMIZE_PUBLIC_MASS_ACTORS, "interpretation": "Public Mass Actor usernames are replaced with deterministic hashes."},
        {"parameter": "strong_association_score", "value": "3*verified_direct_interaction_count + 2*temporal_mass_comment_count", "interpretation": "Shared-video context is not included in strong association score."},
        {"parameter": "contextual_association_score", "value": "strong_association_score + shared_video_context_count", "interpretation": "Descriptive context score, not an influence measure."},
    ]
)


In [24]:
# 27. Visualizations
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 180, "font.size": 9})

palette = {
    "Individual Actor": "#2E7D32",
    "Community Actor": "#1565C0",
    "Mass Actor": "#6A1B9A",
    "Positive": "#2E7D32",
    "Neutral": "#546E7A",
    "Negative": "#C62828",
}

def save_count_bar(series, title, xlabel, ylabel, path, percentage=False, log=False):
    series = series.copy()
    fig, ax = plt.subplots(figsize=(8, 4.8))
    colors = [palette.get(str(idx), "#455A64") for idx in series.index]
    bars = ax.bar(series.index.astype(str), series.values, color=colors)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale("log")
    ax.tick_params(axis="x", rotation=20)
    for bar, value in zip(bars, series.values):
        label = f"{value:.1f}%" if percentage else f"{int(value):,}"
        ax.annotate(label, (bar.get_x() + bar.get_width() / 2, bar.get_height()), ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)

actor_counts = actor_type_summary.set_index("actor_type_primary")["n_accounts"].reindex(VALID_ACTOR_TYPES, fill_value=0)
actor_percent = actor_type_summary.set_index("actor_type_primary")["account_percentage"].reindex(VALID_ACTOR_TYPES, fill_value=0)
comment_counts = actor_type_summary.set_index("actor_type_primary")["n_comments"].reindex(VALID_ACTOR_TYPES, fill_value=0)
comment_percent = actor_type_summary.set_index("actor_type_primary")["comment_percentage"].reindex(VALID_ACTOR_TYPES, fill_value=0)

save_count_bar(actor_counts, "Actor Type Distribution - Accounts (Absolute)", "Actor Type", "Accounts", VIS_DIR / "actor_type_distribution_accounts_absolute.png", log=True)
save_count_bar(actor_percent, "Actor Type Distribution - Accounts (Percentage)", "Actor Type", "Percentage", VIS_DIR / "actor_type_distribution_accounts_percentage.png", percentage=True)
save_count_bar(comment_counts, "Actor Type Distribution - Comments (Absolute)", "Actor Type", "Comments", VIS_DIR / "actor_type_distribution_comments_absolute.png", log=True)
save_count_bar(comment_percent, "Actor Type Distribution - Comments (Percentage)", "Actor Type", "Percentage", VIS_DIR / "actor_type_distribution_comments_percentage.png", percentage=True)

def stacked_bar(table, title, path, ylabel="Accounts"):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    table.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
    ax.set_title(title)
    ax.set_xlabel("Actor Type")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=15)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)

goal_table = pd.crosstab(actors["actor_type_primary"], actors["account_goal_orientation"]).reindex(VALID_ACTOR_TYPES, fill_value=0)
stacked_bar(goal_table, "Actor Type by Account Goal Orientation", VIS_DIR / "actor_type_by_account_goal_orientation.png")
evaluable_goals = actors.loc[~actors["account_goal_orientation"].isin(["Insufficient Text", "Not Observed from Comments"])]
evaluable_goal_table = pd.crosstab(evaluable_goals["actor_type_primary"], evaluable_goals["account_goal_orientation"]).reindex(VALID_ACTOR_TYPES, fill_value=0)
stacked_bar(evaluable_goal_table, "Actor Type by Evaluable Account Goals", VIS_DIR / "actor_type_by_evaluable_account_goals.png")

pooled_sentiment_table = actor_type_goals_pooled.set_index("actor_type_primary")[
    ["pooled_positive_count", "pooled_neutral_count", "pooled_negative_count"]
].reindex(VALID_ACTOR_TYPES, fill_value=0)
pooled_sentiment_table.columns = ["Positive", "Neutral", "Negative"]
stacked_bar(pooled_sentiment_table, "Pooled Sentiment Distribution by Actor Type", VIS_DIR / "pooled_sentiment_distribution_by_actor_type.png", ylabel="Valid Comments")
pooled_goal_counts = actor_type_goals_pooled.set_index("actor_type_primary")["pooled_goal_orientation"].value_counts()
pooled_goal_table = pd.crosstab(actor_type_goals_pooled["actor_type_primary"], actor_type_goals_pooled["pooled_goal_orientation"]).reindex(VALID_ACTOR_TYPES, fill_value=0)
stacked_bar(pooled_goal_table, "Pooled Goal Orientation by Actor Type", VIS_DIR / "pooled_goal_orientation_by_actor_type.png")

exposure_counts = mass_exposure["exposure_level"].value_counts().reindex(VALID_EXPOSURE_LEVELS, fill_value=0)
save_count_bar(exposure_counts, "Mass Actor Observed Association Level", "Observed Association Level", "Accounts", VIS_DIR / "mass_exposure_level_distribution.png", log=True)
exposure_sent_table = pd.crosstab(mass_exposure["exposure_level"], mass_exposure["dominant_sentiment"]).reindex(VALID_EXPOSURE_LEVELS, fill_value=0)
stacked_bar(exposure_sent_table, "Mass Association Level by Dominant Sentiment", VIS_DIR / "mass_exposure_sentiment_distribution.png")

hcc_mass_plot = three_dimensions_typology_hcc.sort_values("mass_fractional_context_weight", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(hcc_mass_plot["hcc_id"].astype(str), hcc_mass_plot["n_mass_unique_associated"], label="Unique primary", color="#1565C0")
ax.plot(hcc_mass_plot["hcc_id"].astype(str), hcc_mass_plot["mass_fractional_context_weight"], marker="o", label="Fractional context weight", color="#EF6C00")
ax.set_title("Mass Association by HCC")
ax.set_xlabel("HCC ID")
ax.set_ylabel("Mass Actor Count / Weight")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
fig.savefig(VIS_DIR / "mass_association_by_hcc.png")
plt.close(fig)

alignment_counts = mass_exposure["sentiment_alignment"].value_counts().reindex(VALID_ALIGNMENT_VALUES, fill_value=0)
save_count_bar(alignment_counts, "Community-Mass Sentiment Alignment", "Alignment Status", "Mass Actors", VIS_DIR / "community_mass_sentiment_alignment.png", log=True)

heat = pd.crosstab(actors["actor_type_primary"], actors["target_scope"]).reindex(VALID_ACTOR_TYPES, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4.8))
im = ax.imshow(heat.values, aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=35, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_title("Three Dimensions Actor-Target-Goal Context")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, int(heat.iat[i, j]), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
fig.tight_layout()
fig.savefig(VIS_DIR / "three_dimensions_actor_target_goal_heatmap.png")
plt.close(fig)

status_counts = mass_exposure["hcc_association_status"].value_counts().reindex(VALID_HCC_ASSOCIATION_STATUS, fill_value=0)
save_count_bar(status_counts, "HCC Association Status Distribution", "HCC Association Status", "Mass Actors", VIS_DIR / "hcc_association_status_distribution.png", log=True)
reply_counts = reply_validation_audit["validation_status"].value_counts()
save_count_bar(reply_counts, "Reply Validation Status Distribution", "Validation Status", "Replies", VIS_DIR / "reply_validation_status_distribution.png", log=True)

hcc_cov_plot = hcc_alignment_coverage.sort_values("alignment_coverage", ascending=False).head(25)
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.bar(hcc_cov_plot["hcc_id"].astype(str), hcc_cov_plot["alignment_coverage"] * 100, color="#00897B")
ax.set_title("Alignment Coverage by HCC")
ax.set_xlabel("HCC ID")
ax.set_ylabel("Coverage (%)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(VIS_DIR / "alignment_coverage_by_hcc.png")
plt.close(fig)

ind_hcc_plot = individual_hcc_association.sort_values("hcc_comment_count", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 5.2))
if not ind_hcc_plot.empty:
    labels = ind_hcc_plot["individual_username"].astype(str) + " -> " + ind_hcc_plot["hcc_id"].astype(str)
    ax.bar(labels, ind_hcc_plot["hcc_comment_count"], color="#5D4037")
    ax.tick_params(axis="x", rotation=45)
ax.set_title("Individual-Community Observed Context")
ax.set_xlabel("Individual-HCC Pair")
ax.set_ylabel("HCC Comments on Creator Videos")
fig.tight_layout()
fig.savefig(VIS_DIR / "individual_hcc_association.png")
plt.close(fig)

print("Visualizations written:", len(list(VIS_DIR.glob("*.png"))))


Visualizations written: 17


In [25]:
# 28. Gephi export
gephi_hcc_nodes_actor_type = hcc_nodes.copy()
gephi_hcc_nodes_actor_type = gephi_hcc_nodes_actor_type.merge(
    actors[
        [
            "username_norm",
            "actor_type_primary",
            "individual_subtype",
            "is_video_creator",
            "is_verified_registry_actor",
            "individual_in_hcc",
            "account_goal_orientation",
            "target_brand_primary",
        ]
    ],
    left_on="username_norm",
    right_on="username_norm",
    how="left",
)
gephi_hcc_nodes_actor_type["actor_type_primary"] = gephi_hcc_nodes_actor_type["actor_type_primary"].fillna("Community Actor")
gephi_hcc_nodes_actor_type.to_csv(GEPHI_DIR / "gephi_hcc_nodes_actor_type.csv", index=False)
shutil.copyfile(HCC_EDGES_PATH, GEPHI_DIR / "gephi_hcc_edges_actor_type.csv")

aggregate_node_records = []
for row in actors.loc[actors["actor_type_primary"].eq("Individual Actor")].itertuples(index=False):
    aggregate_node_records.append(
        {
            "id": f"IND_{getattr(row, 'username_norm')}",
            "label": getattr(row, "public_username"),
            "node_type": "Individual Actor",
            "actor_type_primary": "Individual Actor",
            "n_accounts": 1,
            "n_comments": int(getattr(row, "n_comments")),
            "goal_orientation": getattr(row, "account_goal_orientation"),
            "target_brand": getattr(row, "target_brand_primary"),
        }
    )
for row in hcc_reference.itertuples(index=False):
    aggregate_node_records.append(
        {
            "id": f"HCC_{getattr(row, 'hcc_id')}",
            "label": f"HCC {getattr(row, 'hcc_id')}",
            "node_type": "Community Actor",
            "actor_type_primary": "Community Actor",
            "n_accounts": int(getattr(row, "community_actor_accounts", 0)),
            "n_comments": int(float(getattr(row, "n_comments", 0))),
            "goal_orientation": getattr(row, "goal_orientation"),
            "target_brand": getattr(row, "brand_label_auto"),
        }
    )

edge_records = []
for row in individual_hcc_association.itertuples(index=False):
    individual_norm = normalize_username(getattr(row, "individual_username"))
    edge_records.append(
        {
            "source": f"IND_{individual_norm}",
            "target": f"HCC_{getattr(row, 'hcc_id')}",
            "edge_type": "Observed Creator-Community Context",
            "weight": int(getattr(row, "shared_video_count")),
            "raw_account_count": 1,
            "fractional_account_weight": 1.0,
            "shared_video_count": int(getattr(row, "shared_video_count")),
            "verified_direct_interaction_count": 0,
            "temporal_mass_comment_count": 0,
            "temporal_pair_count": 0,
            "is_ambiguous_association": False,
            "hcc_account_count": int(getattr(row, "hcc_account_count")),
            "hcc_comment_count": int(getattr(row, "hcc_comment_count")),
        }
    )

mass_edges = mass_hcc_association.copy()
if not mass_edges.empty:
    conditions = [
        mass_edges["hcc_association_status"].eq("Unique Direct Association") & mass_edges["is_primary_association"],
        mass_edges["hcc_association_status"].eq("Unique Temporal Association") & mass_edges["is_primary_association"],
        mass_edges["hcc_association_status"].eq("Single Shared-Video Context") & mass_edges["is_primary_association"],
        mass_edges["hcc_association_status"].eq("Multiple / Ambiguous HCC"),
    ]
    choices = [
        "Verified Community-Mass Direct Interaction",
        "Observed Community-Mass Temporal Association",
        "Shared-Video Context Only",
        "Ambiguous Community-Mass Association",
    ]
    mass_edges["edge_type"] = np.select(conditions, choices, default="")
    mass_edges = mass_edges.loc[mass_edges["edge_type"].ne("")].copy()
    mass_edges["fractional_account_weight"] = np.where(
        mass_edges["edge_type"].eq("Ambiguous Community-Mass Association"),
        mass_edges["fractional_context_weight"],
        1.0,
    )
    mass_edges = mass_edges.merge(
        actors[
            [
                "username_norm",
                "dominant_sentiment",
                "n_comments",
                "account_goal_orientation",
                "target_brand_primary",
            ]
        ],
        left_on="mass_username_norm",
        right_on="username_norm",
        how="left",
    )
    mass_edges["dominant_sentiment"] = mass_edges["dominant_sentiment"].fillna("Unknown")
    mass_edges["node_id"] = mass_edges.apply(
        lambda r: f"MASS_HCC_{clean_segment(r['hcc_id'])}_{clean_segment(r['edge_type'])}_{clean_segment(r['dominant_sentiment'])}",
        axis=1,
    )
    mass_edges["node_label"] = (
        "Mass "
        + mass_edges["edge_type"].astype(str)
        + " "
        + mass_edges["hcc_id"].astype(str)
        + " "
        + mass_edges["dominant_sentiment"].astype(str)
    )
    mass_nodes = (
        mass_edges.groupby(["node_id", "node_label", "hcc_id", "edge_type", "dominant_sentiment"], as_index=False)
        .agg(
            n_accounts=("mass_username_norm", "nunique"),
            n_comments=("n_comments", "sum"),
            goal_orientation=("account_goal_orientation", lambda s: s.mode().iloc[0] if not s.mode().empty else ""),
            target_brand=("target_brand_primary", lambda s: s.mode().iloc[0] if not s.mode().empty else ""),
        )
        .rename(columns={"node_id": "id", "node_label": "label"})
    )
    mass_nodes["node_type"] = "Mass Actor Segment"
    mass_nodes["actor_type_primary"] = "Mass Actor"
    aggregate_node_records.extend(mass_nodes.to_dict("records"))

    mass_edge_records = mass_edges.assign(
        source="HCC_" + mass_edges["hcc_id"].astype(str),
        target=mass_edges["node_id"],
        weight=mass_edges["fractional_account_weight"],
        raw_account_count=1,
        temporal_pair_count=mass_edges["temporal_hcc_comment_pair_count"],
        is_ambiguous_association=mass_edges["edge_type"].eq("Ambiguous Community-Mass Association"),
        hcc_account_count=np.nan,
        hcc_comment_count=np.nan,
    )[
        [
            "source",
            "target",
            "edge_type",
            "weight",
            "raw_account_count",
            "fractional_account_weight",
            "shared_video_context_count",
            "verified_direct_interaction_count",
            "temporal_mass_comment_count",
            "temporal_pair_count",
            "is_ambiguous_association",
            "hcc_account_count",
            "hcc_comment_count",
        ]
    ]
    edge_records.extend(mass_edge_records.to_dict("records"))

gephi_three_actor_aggregate_nodes = pd.DataFrame(aggregate_node_records).drop_duplicates("id")
gephi_three_actor_aggregate_edges = pd.DataFrame(edge_records)
if not gephi_three_actor_aggregate_edges.empty:
    group_cols = ["source", "target", "edge_type", "is_ambiguous_association"]
    gephi_three_actor_aggregate_edges = (
        gephi_three_actor_aggregate_edges.groupby(group_cols, as_index=False)
        .agg(
            weight=("weight", "sum"),
            raw_account_count=("raw_account_count", "sum"),
            fractional_account_weight=("fractional_account_weight", "sum"),
            shared_video_count=("shared_video_count", "sum"),
            verified_direct_interaction_count=("verified_direct_interaction_count", "sum"),
            temporal_mass_comment_count=("temporal_mass_comment_count", "sum"),
            temporal_pair_count=("temporal_pair_count", "sum"),
            hcc_account_count=("hcc_account_count", "sum"),
            hcc_comment_count=("hcc_comment_count", "sum"),
        )
    )
gephi_three_actor_aggregate_nodes.to_csv(GEPHI_DIR / "gephi_three_actor_aggregate_nodes.csv", index=False)
gephi_three_actor_aggregate_edges.to_csv(GEPHI_DIR / "gephi_three_actor_aggregate_edges.csv", index=False)

print("Gephi files written:")
print(f"- nodes actor type: {len(gephi_hcc_nodes_actor_type)}")
print(f"- aggregate nodes: {len(gephi_three_actor_aggregate_nodes)}")
print(f"- aggregate edges: {len(gephi_three_actor_aggregate_edges)}")


Gephi files written:
- nodes actor type: 218
- aggregate nodes: 401
- aggregate edges: 502


## 29. Interpretation guide

- Direct Interaction adalah reply langsung yang lolos validasi parent comment.
- Temporal Association adalah komentar Mass Actor yang muncul setelah komentar HCC pada video yang sama
  dalam window 60 menit.
- Shared-Video Context Only hanya menunjukkan konteks video yang sama, bukan asosiasi kuat.
- Sentiment alignment hanya menunjukkan kesamaan kategori sentimen pada data yang evaluable dan tidak
  membuktikan perpindahan opini atau pengaruh.


## 30. Limitations

- Parent comment ID yang tersimpan sebagai notasi ilmiah dapat kehilangan presisi; notebook ini mencoba
  lookup exact dan float-rounded key, lalu menandai sisanya sebagai referensi tidak terverifikasi.
- Actor-level goals tetap membutuhkan minimum tiga komentar valid, sehingga pooled actor-type goals
  disediakan untuk membaca orientasi pesan pada level kelompok.
- Asosiasi ambigu dipertahankan sebagai many-to-many dan tidak dipaksa ke satu primary HCC.


In [26]:
# 31. Final validation report
# Public output tables
account_actor_type = three_dimensions_typology_account.copy()
mass_actor_exposure_public = mass_exposure.copy()
mass_actor_exposure_public["username"] = mass_actor_exposure_public["public_username"]
mass_actor_exposure_public = mass_actor_exposure_public[
    [
        "username",
        "actor_id_anonymous",
        "mass_position_secondary",
        "n_comments",
        "n_valid_text_comments",
        "dominant_sentiment",
        "account_goal_orientation",
        "account_goal_confidence",
        "is_goal_evaluable",
        "goal_evaluation_reason",
        "target_brand_primary",
        "target_scope",
        "exposure_level",
        "hcc_association_status",
        "primary_associated_hcc",
        "associated_hcc_ids",
        "n_associated_hcc",
        "primary_association_basis",
        "is_ambiguous_hcc_association",
        "shared_video_context_count",
        "verified_direct_reply_count",
        "verified_direct_interaction_count",
        "mass_replies_to_hcc_verified_count",
        "hcc_replies_to_mass_verified_count",
        "unverified_reply_reference_count",
        "parent_comment_missing_count",
        "parent_user_mismatch_count",
        "cross_video_parent_mismatch_count",
        "temporal_mass_comment_count",
        "temporal_hcc_comment_pair_count",
        "n_unique_temporal_mass_comments",
        "n_temporal_comment_pairs",
        "strong_association_score",
        "contextual_association_score",
        "sentiment_alignment",
        "hcc_dominant_sentiment",
    ]
]

mass_hcc_association_public = mass_hcc_association[
    [
        "mass_username",
        "mass_actor_id_anonymous",
        "hcc_id",
        "shared_video_context_count",
        "verified_direct_reply_count",
        "verified_direct_interaction_count",
        "mass_replies_to_hcc_verified_count",
        "hcc_replies_to_mass_verified_count",
        "unverified_reply_reference_count",
        "parent_comment_missing_count",
        "parent_user_mismatch_count",
        "cross_video_parent_mismatch_count",
        "temporal_mass_comment_count",
        "temporal_hcc_comment_pair_count",
        "n_unique_temporal_mass_comments",
        "n_temporal_comment_pairs",
        "min_minutes_after_hcc",
        "median_minutes_after_hcc",
        "nearest_prior_hcc_comment_minutes",
        "strong_association_score",
        "contextual_association_score",
        "hcc_association_status",
        "primary_associated_hcc",
        "n_associated_hcc",
        "is_ambiguous_hcc_association",
        "is_primary_association",
        "fractional_context_weight",
    ]
].copy()

output_tables = {
    "account_actor_type.csv": account_actor_type,
    "individual_actor_candidates.csv": individual_actor_candidates,
    "individual_actor_summary.csv": individual_actor_summary,
    "community_actor_summary.csv": community_actor_summary,
    "mass_actor_exposure.csv": mass_actor_exposure_public,
    "mass_hcc_association.csv": mass_hcc_association_public,
    "actor_type_summary.csv": actor_type_summary,
    "actor_type_by_target.csv": actor_type_by_target,
    "actor_type_by_goals.csv": actor_type_by_goals,
    "three_dimensions_typology_account.csv": three_dimensions_typology_account,
    "three_dimensions_typology_hcc.csv": three_dimensions_typology_hcc,
    "method_thresholds.csv": method_thresholds,
    "individual_hcc_association.csv": individual_hcc_association,
    "reply_validation_audit.csv": reply_validation_audit,
    "mass_comment_hcc_alignment.csv": mass_comment_hcc_alignment,
    "actor_type_goals_pooled.csv": actor_type_goals_pooled,
    "hcc_alignment_coverage.csv": hcc_alignment_coverage,
    "hcc_association_ambiguity_summary.csv": hcc_association_ambiguity_summary,
    "actor_universe_audit.csv": actor_universe_audit,
}
for filename, frame in output_tables.items():
    frame.to_csv(TABLE_DIR / filename, index=False)

final_checksums = {name: sha256_file(path) for name, path in PROTECTED_INPUTS.items()}
protected_unchanged = {name: baseline_checksums[name] == final_checksums[name] for name in PROTECTED_INPUTS}

# Deterministic and methodological assertions
if actors["actor_type_primary"].isna().any():
    raise AssertionError("Every actor must have an actor_type_primary.")
if set(actors["actor_type_primary"].unique()) != set(VALID_ACTOR_TYPES):
    raise AssertionError("Actor types must contain exactly the three allowed categories.")
if set(metadata_creators) - set(actors.loc[actors["actor_type_primary"].eq("Individual Actor"), "username_norm"]):
    raise AssertionError("All metadata creators must be Individual Actor.")
if set(verified_registry) - set(actors.loc[actors["actor_type_primary"].eq("Individual Actor"), "username_norm"]):
    raise AssertionError("All Verified registry actors must be Individual Actor.")
candidate_non_creator = set(registry_candidate["username_norm"]) - set(metadata_creators)
if candidate_non_creator & set(actors.loc[actors["actor_type_primary"].eq("Individual Actor"), "username_norm"]):
    raise AssertionError("Candidate non-creators must not become Individual Actor automatically.")
if not actors.loc[actors["is_hcc_member"] & ~actors["actor_type_primary"].eq("Individual Actor"), "actor_type_primary"].eq("Community Actor").all():
    raise AssertionError("Non-individual HCC members must be Community Actor.")
if actors.loc[actors["is_hcc_member"], "actor_type_primary"].eq("Mass Actor").any():
    raise AssertionError("No HCC member may be Mass Actor.")
if actors["username_norm"].duplicated().any():
    raise AssertionError("Duplicate actor rows found.")
if not mass_exposure["hcc_association_status"].isin(VALID_HCC_ASSOCIATION_STATUS).all():
    raise AssertionError("Invalid HCC association status found.")
ambiguous_with_primary = mass_exposure.loc[
    mass_exposure["hcc_association_status"].eq("Multiple / Ambiguous HCC")
    & mass_exposure["primary_associated_hcc"].astype(str).ne("")
]
if not ambiguous_with_primary.empty:
    raise AssertionError("Ambiguous HCC associations must not have a primary HCC.")
direct_status = mass_exposure.loc[mass_exposure["exposure_level"].eq("Direct Interaction")]
if (direct_status["verified_direct_interaction_count"] <= 0).any():
    raise AssertionError("Direct Interaction requires verified direct reply count.")
if mass_hcc_association["temporal_mass_comment_count"].gt(mass_hcc_association["temporal_hcc_comment_pair_count"]).any():
    raise AssertionError("Temporal mass comment count cannot exceed pair count.")
if (hcc_edges.shape[0] != EXPECTED_BASELINES["hcc_edges"]) or (hcc_nodes.shape[0] != EXPECTED_BASELINES["hcc_nodes"]):
    raise AssertionError("Baseline HCC dimensions changed.")
if not all(protected_unchanged.values()):
    changed = [name for name, same in protected_unchanged.items() if not same]
    raise AssertionError(f"Protected inputs changed during notebook execution: {changed}")

created_table_files = sorted(p.name for p in TABLE_DIR.glob("*.csv"))
created_visual_files = sorted(p.name for p in VIS_DIR.glob("*.png"))
created_gephi_files = sorted(p.name for p in GEPHI_DIR.glob("*.csv"))

report = {
    "final_actor_universe": len(actors),
    "commentator_accounts": len(commenters),
    "metadata_creators": len(metadata_creators),
    "creators_without_comments": len(set(metadata_creators) - set(commenters)),
    "verified_registry_noncreator": len(set(verified_registry) - set(metadata_creators)),
    "individual_actor": int(actor_type_counts["Individual Actor"]),
    "community_actor": int(actor_type_counts["Community Actor"]),
    "mass_actor": int(actor_type_counts["Mass Actor"]),
    "creator_also_hcc": len(set(metadata_creators) & hcc_set),
    "lcn_non_hcc_mass": int((actors["mass_position_secondary"] == "LCN Non-HCC").sum()),
    "outside_lcn_mass": int((actors["mass_position_secondary"] == "Outside LCN").sum()),
    "verified_direct_interaction": int(mass_exposure["verified_direct_interaction_count"].sum()),
    "unverified_reply_references": int(mass_exposure["unverified_reply_reference_count"].sum()),
    "temporal_association_mass_comments": int(mass_exposure["temporal_mass_comment_count"].sum()),
    "temporal_association_pairs": int(mass_exposure["temporal_hcc_comment_pair_count"].sum()),
    "shared_video_context_only": int((mass_exposure["exposure_level"] == "Shared-Video Context Only").sum()),
    "no_observed_association": int((mass_exposure["exposure_level"] == "No Observed Community Association").sum()),
    "unique_direct_hcc_association": int((mass_exposure["hcc_association_status"] == "Unique Direct Association").sum()),
    "unique_temporal_hcc_association": int((mass_exposure["hcc_association_status"] == "Unique Temporal Association").sum()),
    "single_shared_context": int((mass_exposure["hcc_association_status"] == "Single Shared-Video Context").sum()),
    "multiple_ambiguous_hcc_association": int((mass_exposure["hcc_association_status"] == "Multiple / Ambiguous HCC").sum()),
    "account_goal_evaluable_coverage": float(actors["is_goal_evaluable"].mean()),
    "account_alignment_coverage": float(account_alignment_coverage),
    "comment_alignment_coverage": float(comment_alignment_coverage),
    "individual_hcc_association_count": len(individual_hcc_association),
    "protected_inputs_unchanged": all(protected_unchanged.values()),
    "table_files_created": len(created_table_files),
    "visual_files_created": len(created_visual_files),
    "gephi_files_created": len(created_gephi_files),
}

print("FINAL VALIDATION REPORT")
for key, value in report.items():
    print(f"- {key}: {value}")
print("\nActor type counts:")
print(actor_type_counts.to_string())
print("\nReply validation:")
print(reply_validation_audit["validation_status"].value_counts().to_string())
print("\nHCC association status:")
print(mass_exposure["hcc_association_status"].value_counts().reindex(VALID_HCC_ASSOCIATION_STATUS, fill_value=0).to_string())
print("\nPooled goals per actor type:")
print(actor_type_goals_pooled[["actor_type_primary", "pooled_goal_orientation", "pooled_dominant_sentiment", "n_valid_comments"]].to_string(index=False))
print("\nOutput tables:")
print("\n".join(created_table_files))
print("\nOutput visualizations:")
print("\n".join(created_visual_files))
print("\nOutput Gephi:")
print("\n".join(created_gephi_files))


FINAL VALIDATION REPORT
- final_actor_universe: 26427
- commentator_accounts: 26424
- metadata_creators: 43
- creators_without_comments: 3
- verified_registry_noncreator: 0
- individual_actor: 43
- community_actor: 218
- mass_actor: 26166
- creator_also_hcc: 0
- lcn_non_hcc_mass: 506
- outside_lcn_mass: 25660
- verified_direct_interaction: 63
- unverified_reply_references: 7739
- temporal_association_mass_comments: 2825
- temporal_association_pairs: 5254
- shared_video_context_only: 23864
- no_observed_association: 0
- unique_direct_hcc_association: 28
- unique_temporal_hcc_association: 1871
- single_shared_context: 1396
- multiple_ambiguous_hcc_association: 22871
- account_goal_evaluable_coverage: 0.04817043175540167
- account_alignment_coverage: 0.0022548345180768937
- comment_alignment_coverage: 0.45602493074792244
- individual_hcc_association_count: 186
- protected_inputs_unchanged: True
- table_files_created: 19
- visual_files_created: 17
- gephi_files_created: 4

Actor type count

## 32. Actor Type Visualization in Gephi

Section ini menambahkan ekspor node-edge Gephi untuk visualisasi **Type Actor** pada RM2 tanpa menghitung ulang LCN, Louvain, FSA_V, HCC membership, edge RM1, model sentimen, goals, maupun temporal RM1.

Prinsip utamanya:

- `actor_type_primary` mencakup seluruh actor universe: `Individual Actor`, `Community Actor`, dan `Mass Actor`.
- Prioritas klasifikasi tetap: `Individual Actor > Community Actor > Mass Actor`.
- `network_position` dipisahkan dari `actor_type_primary`: `HCC`, `LCN Non-HCC`, atau `Outside LCN`.
- Mass Actor adalah residual seluruh actor universe setelah Individual Actor dan Community Actor dikeluarkan, bukan hanya akun LCN.
- Visualisasi Gephi utama hanya memakai subset akun yang masuk LCN karena akun di luar LCN tidak mempunyai edge koordinasi yang memenuhi kriteria jaringan.

Caption standar: Visualisasi hanya mencakup akun yang masuk Latent Coordination Network. Mass Actor di luar LCN tidak ditampilkan karena tidak mempunyai edge koordinasi yang memenuhi kriteria pembentukan jaringan. Seluruh Mass Actor tetap dihitung dalam ringkasan statistik actor universe.

In [27]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RM2_ACTOR_DIR = ROOT / "output" / "rm2_actor_type"
ACTOR_TABLE_DIR = RM2_ACTOR_DIR / "tables"
ACTOR_VIS_DIR = RM2_ACTOR_DIR / "visualisasi"
ACTOR_GEPHI_DIR = RM2_ACTOR_DIR / "gephi"
ACTOR_TABLE_DIR.mkdir(parents=True, exist_ok=True)
ACTOR_VIS_DIR.mkdir(parents=True, exist_ok=True)
ACTOR_GEPHI_DIR.mkdir(parents=True, exist_ok=True)

ACCOUNT_ACTOR_TYPE_PATH = ACTOR_TABLE_DIR / "account_actor_type.csv"
LCN_NODES_ACTUAL_PATH = ROOT / "output" / "gephi" / "gephi_lcn_nodes.csv"
LCN_EDGES_ACTUAL_PATH = ROOT / "output" / "gephi" / "gephi_lcn_edges.csv"
HCC_NODES_ACTUAL_PATH = ROOT / "output" / "gephi" / "gephi_hcc_nodes.csv"
GEPI_NODE_OUT = ACTOR_GEPHI_DIR / "gephi_lcn_nodes_actor_type.csv"
GEPI_EDGE_OUT = ACTOR_GEPHI_DIR / "gephi_lcn_edges_actor_type.csv"

VALID_ACTOR_TYPES_ORDERED = ["Individual Actor", "Community Actor", "Mass Actor"]
VALID_NETWORK_POSITIONS = ["HCC", "LCN Non-HCC", "Outside LCN"]
PAIR_ORDER = {"Individual Actor": 0, "Community Actor": 1, "Mass Actor": 2}
PAIR_SHORT = {"Individual Actor": "Individual", "Community Actor": "Community", "Mass Actor": "Mass"}
NETWORK_POSITION_ORDER = {"HCC": 0, "LCN Non-HCC": 1, "Outside LCN": 2}
VALID_EDGE_PAIRS = [
    "Individual–Individual",
    "Individual–Community",
    "Individual–Mass",
    "Community–Community",
    "Community–Mass",
    "Mass–Mass",
]
ACTOR_COLORS = {"Individual Actor": "#E69F00", "Community Actor": "#009E73", "Mass Actor": "#8A8A8A"}
POSITION_COLORS = {"HCC": "#009E73", "LCN Non-HCC": "#56B4E9", "Outside LCN": "#BDBDBD"}


def _read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File input wajib tidak ditemukan: {path}")
    return pd.read_csv(path)


def _as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


def _network_position(row) -> str:
    if bool(row["is_hcc_member"]):
        return "HCC"
    if bool(row["is_lcn_member"]):
        return "LCN Non-HCC"
    return "Outside LCN"


def _dominant_goal(series: pd.Series) -> str:
    cleaned = series.fillna("").astype(str).str.strip()
    cleaned = cleaned[cleaned.ne("") & cleaned.str.lower().ne("nan")]
    return "Not available" if cleaned.empty else str(cleaned.value_counts().idxmax())


def _safe_ratio(numerator: float, denominator: float) -> float:
    return round(float(numerator) / float(denominator), 4) if denominator else 0.0


def _canonical_actor_pair(a: str, b: str) -> str:
    ordered = sorted([a, b], key=lambda value: PAIR_ORDER.get(value, 99))
    return f"{PAIR_SHORT.get(ordered[0], ordered[0])}–{PAIR_SHORT.get(ordered[1], ordered[1])}"


def _canonical_position_pair(a: str, b: str) -> str:
    ordered = sorted([a, b], key=lambda value: NETWORK_POSITION_ORDER.get(value, 99))
    return f"{ordered[0]}–{ordered[1]}"


def _edge_struct_hash(df: pd.DataFrame, columns: list[str]) -> str:
    normalized = df[columns].copy()
    if "source" in columns and "target" in columns:
        normalized = normalized.sort_values(["source", "target"]).reset_index(drop=True)
    payload = normalized.to_csv(index=False, lineterminator="\n", float_format="%.12g").encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def _file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def _write_bar_with_zoom(df, value_col, title, ylabel, output_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.1, 1]})
    ordered = df.set_index("actor_type_primary").reindex(VALID_ACTOR_TYPES_ORDERED).reset_index()
    colors = [ACTOR_COLORS[t] for t in ordered["actor_type_primary"]]
    axes[0].bar(ordered["actor_type_primary"], ordered[value_col], color=colors)
    axes[0].set_yscale("log")
    axes[0].set_title("Seluruh actor type (skala log)")
    axes[0].set_ylabel(ylabel)
    axes[0].tick_params(axis="x", rotation=20)
    for x, y in zip(ordered["actor_type_primary"], ordered[value_col]):
        axes[0].text(x, max(float(y), 1) * 1.05, f"{int(y):,}", ha="center", va="bottom", fontsize=8)

    zoom = ordered[ordered["actor_type_primary"].isin(["Individual Actor", "Community Actor"])]
    axes[1].bar(zoom["actor_type_primary"], zoom[value_col], color=[ACTOR_COLORS[t] for t in zoom["actor_type_primary"]])
    axes[1].set_title("Zoom Individual–Community")
    axes[1].tick_params(axis="x", rotation=20)
    for x, y in zip(zoom["actor_type_primary"], zoom[value_col]):
        axes[1].text(x, float(y) * 1.02 if y else 0.2, f"{int(y):,}", ha="center", va="bottom", fontsize=8)
    fig.suptitle(title, fontsize=13, fontweight="bold")
    fig.text(0.01, 0.01, "Catatan: panel kiri memakai skala log karena Mass Actor sangat dominan.", fontsize=8)
    fig.tight_layout(rect=[0, 0.04, 1, 0.95])
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def _write_horizontal_bar(summary, value_col, title, xlabel, output_path):
    plot_df = summary.set_index("actor_type_pair").reindex(VALID_EDGE_PAIRS).reset_index()
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(plot_df["actor_type_pair"], plot_df[value_col], color="#607D8B")
    ax.invert_yaxis()
    ax.set_title(title, fontsize=12.5, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Actor type pair")
    max_value = max(plot_df[value_col].max(), 1)
    for bar, value in zip(bars, plot_df[value_col]):
        label = f"{value:,.2f}" if isinstance(value, float) and not float(value).is_integer() else f"{int(value):,}"
        ax.text(bar.get_width() + max_value * 0.01, bar.get_y() + bar.get_height() / 2, label, va="center", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


input_hashes_before = {
    "gephi_lcn_nodes.csv": _file_sha256(LCN_NODES_ACTUAL_PATH),
    "gephi_lcn_edges.csv": _file_sha256(LCN_EDGES_ACTUAL_PATH),
    "gephi_hcc_nodes.csv": _file_sha256(HCC_NODES_ACTUAL_PATH),
}

account_source = _read_csv_required(ACCOUNT_ACTOR_TYPE_PATH)
lcn_nodes_source = _read_csv_required(LCN_NODES_ACTUAL_PATH)
lcn_edges_source = _read_csv_required(LCN_EDGES_ACTUAL_PATH)
hcc_nodes_source = _read_csv_required(HCC_NODES_ACTUAL_PATH)

account = account_source.copy()
actor_universe_count_before = len(account)
actor_universe_unique_before = account["username"].astype(str).nunique()
for col in ["is_hcc_member", "is_lcn_member", "is_video_creator", "is_verified_registry_actor"]:
    account[col] = _as_bool(account[col])
account["network_position"] = account.apply(_network_position, axis=1)
if "network_position" not in account_source.columns:
    cols = list(account.columns)
    cols.insert(cols.index("actor_type_primary") + 1, cols.pop(cols.index("network_position")))
    account = account[cols]
account.to_csv(ACCOUNT_ACTOR_TYPE_PATH, index=False)

pd.DataFrame([
    {"input_role": "LCN nodes", "path_actual": str(LCN_NODES_ACTUAL_PATH.relative_to(ROOT)), "n_rows": len(lcn_nodes_source), "notes": "Node LCN sumber RM1; hanya dibaca."},
    {"input_role": "LCN edges", "path_actual": str(LCN_EDGES_ACTUAL_PATH.relative_to(ROOT)), "n_rows": len(lcn_edges_source), "notes": "Edge LCN sumber RM1; hanya dibaca."},
    {"input_role": "HCC nodes", "path_actual": str(HCC_NODES_ACTUAL_PATH.relative_to(ROOT)), "n_rows": len(hcc_nodes_source), "notes": "Node HCC final sumber RM1; hanya dibaca."},
    {"input_role": "Actor attributes", "path_actual": str(ACCOUNT_ACTOR_TYPE_PATH.relative_to(ROOT)), "n_rows": len(account), "notes": "Sumber utama atribut actor type RM2."},
]).to_csv(ACTOR_TABLE_DIR / "actor_type_gephi_input_audit.csv", index=False)

# A. Actor universe summary.
universe_rows = []
for actor_type in VALID_ACTOR_TYPES_ORDERED:
    subset = account[account["actor_type_primary"].eq(actor_type)].copy()
    n_accounts = int(len(subset))
    n_comments = int(pd.to_numeric(subset["n_comments"], errors="coerce").fillna(0).sum())
    sentiment_den = pd.to_numeric(subset[["positive_count", "neutral_count", "negative_count"]].stack(), errors="coerce").fillna(0).sum()
    universe_rows.append({
        "actor_type_primary": actor_type,
        "n_accounts": n_accounts,
        "account_percentage": round(_safe_ratio(n_accounts * 100, len(account)), 3),
        "n_accounts_with_comments": int((pd.to_numeric(subset["n_comments"], errors="coerce").fillna(0) > 0).sum()),
        "n_comments": n_comments,
        "comment_percentage": 0.0,
        "mean_comments_per_account": round(float(pd.to_numeric(subset["n_comments"], errors="coerce").fillna(0).mean()), 3) if n_accounts else 0.0,
        "median_comments_per_account": round(float(pd.to_numeric(subset["n_comments"], errors="coerce").fillna(0).median()), 3) if n_accounts else 0.0,
        "n_hcc_members": int(subset["is_hcc_member"].sum()),
        "n_lcn_members": int(subset["is_lcn_member"].sum()),
        "n_outside_lcn": int(subset["network_position"].eq("Outside LCN").sum()),
        "positive_ratio": _safe_ratio(pd.to_numeric(subset["positive_count"], errors="coerce").fillna(0).sum(), sentiment_den),
        "neutral_ratio": _safe_ratio(pd.to_numeric(subset["neutral_count"], errors="coerce").fillna(0).sum(), sentiment_den),
        "negative_ratio": _safe_ratio(pd.to_numeric(subset["negative_count"], errors="coerce").fillna(0).sum(), sentiment_den),
        "dominant_goal_orientation": _dominant_goal(subset["account_goal_orientation"]),
    })
actor_type_universe_summary = pd.DataFrame(universe_rows)
total_comments_universe = actor_type_universe_summary["n_comments"].sum()
actor_type_universe_summary["comment_percentage"] = (actor_type_universe_summary["n_comments"] / total_comments_universe * 100).fillna(0).round(3)
total_sentiment_den = pd.to_numeric(account[["positive_count", "neutral_count", "negative_count"]].stack(), errors="coerce").fillna(0).sum()
total_row = {
    "actor_type_primary": "Total",
    "n_accounts": int(len(account)),
    "account_percentage": 100.0,
    "n_accounts_with_comments": int((pd.to_numeric(account["n_comments"], errors="coerce").fillna(0) > 0).sum()),
    "n_comments": int(total_comments_universe),
    "comment_percentage": 100.0,
    "mean_comments_per_account": round(float(pd.to_numeric(account["n_comments"], errors="coerce").fillna(0).mean()), 3),
    "median_comments_per_account": round(float(pd.to_numeric(account["n_comments"], errors="coerce").fillna(0).median()), 3),
    "n_hcc_members": int(account["is_hcc_member"].sum()),
    "n_lcn_members": int(account["is_lcn_member"].sum()),
    "n_outside_lcn": int(account["network_position"].eq("Outside LCN").sum()),
    "positive_ratio": _safe_ratio(pd.to_numeric(account["positive_count"], errors="coerce").fillna(0).sum(), total_sentiment_den),
    "neutral_ratio": _safe_ratio(pd.to_numeric(account["neutral_count"], errors="coerce").fillna(0).sum(), total_sentiment_den),
    "negative_ratio": _safe_ratio(pd.to_numeric(account["negative_count"], errors="coerce").fillna(0).sum(), total_sentiment_den),
    "dominant_goal_orientation": _dominant_goal(account["account_goal_orientation"]),
}
actor_type_universe_summary = pd.concat([actor_type_universe_summary, pd.DataFrame([total_row])], ignore_index=True)
actor_type_universe_summary.to_csv(ACTOR_TABLE_DIR / "actor_type_universe_summary.csv", index=False)

# F. Actor type x network position matrix.
position_matrix = pd.crosstab(account["actor_type_primary"], account["network_position"]).reindex(VALID_ACTOR_TYPES_ORDERED, fill_value=0).reindex(columns=VALID_NETWORK_POSITIONS, fill_value=0)
position_matrix["Total"] = position_matrix.sum(axis=1)
position_matrix = position_matrix.reset_index()
total_position_row = {"actor_type_primary": "Total"}
for col in VALID_NETWORK_POSITIONS + ["Total"]:
    total_position_row[col] = int(position_matrix[col].sum())
actor_type_network_position_matrix = pd.concat([position_matrix, pd.DataFrame([total_position_row])], ignore_index=True)
actor_type_network_position_matrix.to_csv(ACTOR_TABLE_DIR / "actor_type_network_position_matrix.csv", index=False)

# C. Node Gephi actor type.
lcn_nodes = lcn_nodes_source.copy()
lcn_nodes["id"] = lcn_nodes["id"].astype(str)
lcn_nodes["label"] = lcn_nodes["label"].astype(str)
account_for_lcn = account.copy()
account_for_lcn["display_name"] = account_for_lcn["display_name"].astype(str)
if account_for_lcn["display_name"].duplicated().any():
    dup = account_for_lcn.loc[account_for_lcn["display_name"].duplicated(), "display_name"].head(10).tolist()
    raise AssertionError(f"display_name duplikat pada account_actor_type: {dup}")

attribute_cols = [
    "display_name", "actor_type_primary", "network_position", "is_hcc_member", "is_lcn_member",
    "individual_subtype", "is_video_creator", "is_verified_registry_actor", "n_comments",
    "dominant_sentiment", "target_brand_primary", "target_brand_label", "target_scope",
    "account_goal_orientation", "account_goal_confidence", "exposure_level", "hcc_association_status",
    "strong_association_score", "contextual_association_score",
]
gephi_nodes = lcn_nodes.merge(account_for_lcn[attribute_cols], left_on="id", right_on="display_name", how="left", validate="one_to_one")
if gephi_nodes["actor_type_primary"].isna().any():
    missing = gephi_nodes.loc[gephi_nodes["actor_type_primary"].isna(), "id"].head(10).tolist()
    raise AssertionError(f"Atribut actor type tidak ditemukan untuk node LCN: {missing}")
gephi_nodes = gephi_nodes.drop(columns=["display_name"])
gephi_nodes["is_hcc_member"] = _as_bool(gephi_nodes["is_hcc_member"])
gephi_nodes["is_lcn_member"] = _as_bool(gephi_nodes["is_lcn_member"])
if gephi_nodes["network_position"].eq("Outside LCN").any():
    raise AssertionError("Node Outside LCN tidak boleh masuk Gephi LCN actor type.")

gephi_nodes_out = gephi_nodes.rename(columns={"id": "Id", "label": "Label"}).copy()
minimal_node_cols = [
    "Id", "Label", "actor_type_primary", "network_position", "is_hcc_member", "community",
    "individual_subtype", "is_video_creator", "degree", "weighted_degree", "betweenness", "n_comments",
    "target_brand_primary", "target_brand_label", "target_scope", "dominant_sentiment",
    "account_goal_orientation", "account_goal_confidence", "exposure_level", "hcc_association_status",
    "strong_association_score", "contextual_association_score",
]
gephi_nodes_out = gephi_nodes_out[minimal_node_cols + [c for c in gephi_nodes_out.columns if c not in minimal_node_cols]]
gephi_nodes_out.to_csv(GEPI_NODE_OUT, index=False)

# B. LCN actor type summary.
lcn_summary_grouped = gephi_nodes.groupby(["actor_type_primary", "network_position"], dropna=False).agg(
    n_nodes=("id", "nunique"),
    n_comments=("n_comments", "sum"),
    mean_degree=("degree", "mean"),
    median_degree=("degree", "median"),
    mean_weighted_degree=("weighted_degree", "mean"),
    median_weighted_degree=("weighted_degree", "median"),
    mean_betweenness=("betweenness", "mean"),
    max_betweenness=("betweenness", "max"),
).reset_index()
lcn_grid = pd.DataFrame([
    {"actor_type_primary": "Individual Actor", "network_position": "HCC"},
    {"actor_type_primary": "Individual Actor", "network_position": "LCN Non-HCC"},
    {"actor_type_primary": "Community Actor", "network_position": "HCC"},
    {"actor_type_primary": "Mass Actor", "network_position": "LCN Non-HCC"},
])
lcn_actor_type_summary = lcn_grid.merge(lcn_summary_grouped, how="left", on=["actor_type_primary", "network_position"])
for col in ["n_nodes", "n_comments"]:
    lcn_actor_type_summary[col] = lcn_actor_type_summary[col].fillna(0).astype(int)
for col in ["mean_degree", "median_degree", "mean_weighted_degree", "median_weighted_degree", "mean_betweenness", "max_betweenness"]:
    lcn_actor_type_summary[col] = lcn_actor_type_summary[col].fillna(0).round(6)
lcn_actor_type_summary["node_percentage_within_lcn"] = (lcn_actor_type_summary["n_nodes"] / len(lcn_nodes) * 100).round(3)
lcn_actor_type_summary = lcn_actor_type_summary[
    ["actor_type_primary", "network_position", "n_nodes", "node_percentage_within_lcn", "n_comments",
     "mean_degree", "median_degree", "mean_weighted_degree", "median_weighted_degree", "mean_betweenness", "max_betweenness"]
]
lcn_actor_type_summary.to_csv(ACTOR_TABLE_DIR / "lcn_actor_type_summary.csv", index=False)

# D. Edge Gephi actor type.
lcn_edges = lcn_edges_source.copy()
source_struct_cols = list(lcn_edges.columns)
source_edge_hash = _edge_struct_hash(lcn_edges, source_struct_cols)
node_actor = gephi_nodes.set_index("id")["actor_type_primary"].to_dict()
node_position = gephi_nodes.set_index("id")["network_position"].to_dict()
node_is_hcc = gephi_nodes.set_index("id")["is_hcc_member"].to_dict()
node_ids = set(gephi_nodes["id"].astype(str))
edge_missing_sources = sorted(set(lcn_edges["source"].astype(str)) - node_ids)
edge_missing_targets = sorted(set(lcn_edges["target"].astype(str)) - node_ids)
if edge_missing_sources or edge_missing_targets:
    raise AssertionError(f"Endpoint edge hilang dari node output. Source: {edge_missing_sources[:5]}, Target: {edge_missing_targets[:5]}")

edge_out = pd.DataFrame({
    "Source": lcn_edges["source"].astype(str),
    "Target": lcn_edges["target"].astype(str),
    "Type": "Undirected",
    "Weight": pd.to_numeric(lcn_edges["final_weight"], errors="coerce"),
})
for col in [c for c in lcn_edges.columns if c not in ["source", "target"]]:
    edge_out[col] = lcn_edges[col]
edge_out["source_actor_type"] = edge_out["Source"].map(node_actor)
edge_out["target_actor_type"] = edge_out["Target"].map(node_actor)
edge_out["actor_type_pair"] = [_canonical_actor_pair(a, b) for a, b in zip(edge_out["source_actor_type"], edge_out["target_actor_type"])]
edge_out["source_network_position"] = edge_out["Source"].map(node_position)
edge_out["target_network_position"] = edge_out["Target"].map(node_position)
edge_out["network_position_pair"] = [_canonical_position_pair(a, b) for a, b in zip(edge_out["source_network_position"], edge_out["target_network_position"])]
edge_out["source_is_hcc"] = edge_out["Source"].map(node_is_hcc).astype(bool)
edge_out["target_is_hcc"] = edge_out["Target"].map(node_is_hcc).astype(bool)
edge_out.to_csv(GEPI_EDGE_OUT, index=False)
edge_output_struct = edge_out.rename(columns={"Source": "source", "Target": "target"})[source_struct_cols].copy()
output_edge_hash = _edge_struct_hash(edge_output_struct, source_struct_cols)

edge_summary_grouped = edge_out.groupby("actor_type_pair", dropna=False).agg(
    n_edges=("Source", "count"),
    total_weight=("Weight", "sum"),
    mean_weight=("Weight", "mean"),
    median_weight=("Weight", "median"),
    max_weight=("Weight", "max"),
    n_unique_source_nodes=("Source", "nunique"),
    n_unique_target_nodes=("Target", "nunique"),
).reset_index()
edge_summary_grouped["n_unique_nodes_involved"] = edge_summary_grouped["actor_type_pair"].map(
    lambda pair: len(set(edge_out.loc[edge_out["actor_type_pair"].eq(pair), "Source"]) | set(edge_out.loc[edge_out["actor_type_pair"].eq(pair), "Target"]))
)
edge_summary = pd.DataFrame({"actor_type_pair": VALID_EDGE_PAIRS}).merge(edge_summary_grouped, how="left", on="actor_type_pair")
for col in ["n_edges", "n_unique_source_nodes", "n_unique_target_nodes", "n_unique_nodes_involved"]:
    edge_summary[col] = edge_summary[col].fillna(0).astype(int)
for col in ["total_weight", "mean_weight", "median_weight", "max_weight"]:
    edge_summary[col] = edge_summary[col].fillna(0).round(6)
edge_summary["edge_percentage"] = (edge_summary["n_edges"] / max(len(edge_out), 1) * 100).round(3)
edge_summary["weight_percentage"] = (edge_summary["total_weight"] / max(edge_summary["total_weight"].sum(), 1) * 100).round(3)
edge_summary = edge_summary[
    ["actor_type_pair", "n_edges", "edge_percentage", "total_weight", "weight_percentage", "mean_weight", "median_weight",
     "max_weight", "n_unique_source_nodes", "n_unique_target_nodes", "n_unique_nodes_involved"]
]
edge_summary.to_csv(ACTOR_TABLE_DIR / "lcn_actor_type_edge_summary.csv", index=False)

edge_integrity_audit = pd.DataFrame([
    {"metric": "edge_count", "source_value": len(lcn_edges), "output_value": len(edge_out), "passed": len(lcn_edges) == len(edge_out), "notes": "Jumlah edge output harus sama dengan edge LCN sumber."},
    {"metric": "missing_source_endpoints", "source_value": 0, "output_value": len(edge_missing_sources), "passed": len(edge_missing_sources) == 0, "notes": "Semua Source harus tersedia pada node output."},
    {"metric": "missing_target_endpoints", "source_value": 0, "output_value": len(edge_missing_targets), "passed": len(edge_missing_targets) == 0, "notes": "Semua Target harus tersedia pada node output."},
    {"metric": "self_loop_count", "source_value": int((lcn_edges["source"].astype(str) == lcn_edges["target"].astype(str)).sum()), "output_value": int((edge_out["Source"].astype(str) == edge_out["Target"].astype(str)).sum()), "passed": int((edge_out["Source"].astype(str) == edge_out["Target"].astype(str)).sum()) == int((lcn_edges["source"].astype(str) == lcn_edges["target"].astype(str)).sum()), "notes": "Tidak ada self-loop baru."},
    {"metric": "weight_values_unchanged", "source_value": "final_weight", "output_value": "Weight", "passed": bool(np.allclose(pd.to_numeric(lcn_edges["final_weight"], errors="coerce"), pd.to_numeric(edge_out["Weight"], errors="coerce"), equal_nan=True)), "notes": "Weight output diambil dari final_weight RM1 tanpa perubahan nilai."},
    {"metric": "edge_structural_hash", "source_value": source_edge_hash, "output_value": output_edge_hash, "passed": source_edge_hash == output_edge_hash, "notes": "Hash source/target/evidence/final_weight sama setelah sorting deterministik."},
])
edge_integrity_audit.to_csv(ACTOR_TABLE_DIR / "gephi_lcn_edge_integrity_audit.csv", index=False)

# Visualizations.
plot_universe = actor_type_universe_summary[actor_type_universe_summary["actor_type_primary"].ne("Total")].copy()
_write_bar_with_zoom(plot_universe, "n_accounts", "Jumlah Akun per Primary Actor Type (Actor Universe)", "Jumlah akun", ACTOR_VIS_DIR / "actor_type_account_count.png")
_write_bar_with_zoom(plot_universe, "n_comments", "Jumlah Komentar per Primary Actor Type (Actor Universe)", "Jumlah komentar", ACTOR_VIS_DIR / "actor_type_comment_count.png")
_write_horizontal_bar(edge_summary, "n_edges", "Jumlah Edge LCN menurut Actor-Type Pair", "Jumlah edge", ACTOR_VIS_DIR / "lcn_edge_count_by_actor_type_pair.png")
_write_horizontal_bar(edge_summary, "total_weight", "Total Weight Edge LCN menurut Actor-Type Pair", "Total weight", ACTOR_VIS_DIR / "lcn_edge_weight_by_actor_type_pair.png")

matrix_plot = actor_type_network_position_matrix[actor_type_network_position_matrix["actor_type_primary"].ne("Total")].copy()
counts = matrix_plot.set_index("actor_type_primary")[VALID_NETWORK_POSITIONS].reindex(VALID_ACTOR_TYPES_ORDERED)
percent = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
fig, ax = plt.subplots(figsize=(10, 5.2))
left = np.zeros(len(percent))
for position in VALID_NETWORK_POSITIONS:
    values = percent[position].values
    bars = ax.barh(percent.index, values, left=left, color=POSITION_COLORS[position], label=position)
    for idx, (bar, value, count) in enumerate(zip(bars, values, counts[position].values)):
        if value >= 4:
            ax.text(left[idx] + value / 2, bar.get_y() + bar.get_height() / 2, f"{count:,}\n{value:.1f}%", ha="center", va="center", fontsize=8)
    left += values
ax.set_xlim(0, 100)
ax.set_xlabel("Persentase akun dalam actor type (%)")
ax.set_ylabel("Primary actor type")
ax.set_title("Actor Type x Network Position (Actor Universe)", fontsize=12.5, fontweight="bold")
ax.legend(title="Network position", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.text(0.01, 0.01, "Mass Actor di luar LCN tetap dihitung dalam universe, tetapi tidak masuk Gephi LCN.", fontsize=8)
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(ACTOR_VIS_DIR / "actor_type_network_position_stacked.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# I. Validation report.
input_hashes_after = {
    "gephi_lcn_nodes.csv": _file_sha256(LCN_NODES_ACTUAL_PATH),
    "gephi_lcn_edges.csv": _file_sha256(LCN_EDGES_ACTUAL_PATH),
    "gephi_hcc_nodes.csv": _file_sha256(HCC_NODES_ACTUAL_PATH),
}
png_files = [
    ACTOR_VIS_DIR / "actor_type_account_count.png",
    ACTOR_VIS_DIR / "actor_type_comment_count.png",
    ACTOR_VIS_DIR / "lcn_edge_count_by_actor_type_pair.png",
    ACTOR_VIS_DIR / "lcn_edge_weight_by_actor_type_pair.png",
    ACTOR_VIS_DIR / "actor_type_network_position_stacked.png",
]
individual_rule = account["is_video_creator"] | account["is_verified_registry_actor"]
community_rule = account["is_hcc_member"] & ~individual_rule
mass_rule = ~individual_rule & ~account["is_hcc_member"]
hcc_ids = set(hcc_nodes_source["id"].astype(str))
lcn_ids = set(lcn_nodes_source["id"].astype(str))
node_out_ids = set(gephi_nodes_out["Id"].astype(str))
edge_endpoint_ids = set(edge_out["Source"].astype(str)) | set(edge_out["Target"].astype(str))
validation_rows = [
    ("actor_universe_row_count", actor_universe_count_before, len(account), actor_universe_count_before == len(account), "Penambahan network_position tidak mengubah jumlah baris actor universe."),
    ("actor_universe_unique_accounts", actor_universe_unique_before, account["username"].astype(str).nunique(), actor_universe_unique_before == account["username"].astype(str).nunique(), "Jumlah akun unik tidak berubah."),
    ("single_actor_type_per_account", 0, int(account.duplicated("username").sum()), int(account.duplicated("username").sum()) == 0, "Setiap akun hanya satu baris dan satu actor_type_primary."),
    ("actor_type_not_empty", 0, int(account["actor_type_primary"].isna().sum() + account["actor_type_primary"].astype(str).str.strip().eq("").sum()), not account["actor_type_primary"].isna().any() and not account["actor_type_primary"].astype(str).str.strip().eq("").any(), "Tidak ada actor type kosong."),
    ("individual_priority", 0, int((individual_rule & ~account["actor_type_primary"].eq("Individual Actor")).sum()), int((individual_rule & ~account["actor_type_primary"].eq("Individual Actor")).sum()) == 0, "Video creator/verified registry selalu Individual Actor."),
    ("community_actor_rule", 0, int((community_rule & ~account["actor_type_primary"].eq("Community Actor")).sum()), int((community_rule & ~account["actor_type_primary"].eq("Community Actor")).sum()) == 0, "HCC non-individual menjadi Community Actor."),
    ("mass_actor_residual_rule", int(mass_rule.sum()), int(account["actor_type_primary"].eq("Mass Actor").sum()), int(mass_rule.sum()) == int(account["actor_type_primary"].eq("Mass Actor").sum()), "Mass Actor adalah residual actor universe."),
    ("mass_actor_lcn_or_outside", "LCN Non-HCC/Outside LCN", int(account[account["actor_type_primary"].eq("Mass Actor")]["network_position"].isin(["LCN Non-HCC", "Outside LCN"]).sum()), account[account["actor_type_primary"].eq("Mass Actor")]["network_position"].isin(["LCN Non-HCC", "Outside LCN"]).all(), "Mass Actor dapat berada di LCN Non-HCC atau Outside LCN."),
    ("community_actor_position_hcc", 0, int((account["actor_type_primary"].eq("Community Actor") & ~account["network_position"].eq("HCC")).sum()), int((account["actor_type_primary"].eq("Community Actor") & ~account["network_position"].eq("HCC")).sum()) == 0, "Seluruh Community Actor berada di HCC."),
    ("hcc_subset_lcn", len(hcc_ids), len(hcc_ids & lcn_ids), hcc_ids.issubset(lcn_ids), "Seluruh node HCC merupakan subset node LCN."),
    ("gephi_node_count_equals_lcn", len(lcn_nodes_source), len(gephi_nodes_out), len(lcn_nodes_source) == len(gephi_nodes_out), "Node Gephi actor type sama dengan node LCN sumber."),
    ("gephi_node_id_set_equals_lcn", len(lcn_ids), len(node_out_ids & lcn_ids), node_out_ids == lcn_ids, "Set node output sama dengan set node LCN sumber."),
    ("no_outside_lcn_in_gephi", 0, int(gephi_nodes_out["network_position"].eq("Outside LCN").sum()), int(gephi_nodes_out["network_position"].eq("Outside LCN").sum()) == 0, "Tidak ada node Outside LCN dalam Gephi LCN."),
    ("edge_endpoints_available", len(edge_endpoint_ids), len(edge_endpoint_ids & node_out_ids), edge_endpoint_ids.issubset(node_out_ids), "Semua endpoint edge tersedia pada node output."),
    ("edge_count_unchanged", len(lcn_edges_source), len(edge_out), len(lcn_edges_source) == len(edge_out), "Jumlah edge tidak berubah."),
    ("edge_weight_unchanged", "final_weight", "Weight", bool(np.allclose(pd.to_numeric(lcn_edges_source["final_weight"], errors="coerce"), pd.to_numeric(edge_out["Weight"], errors="coerce"), equal_nan=True)), "Weight edge tidak berubah dari final_weight RM1."),
    ("edge_structural_hash_unchanged", source_edge_hash, output_edge_hash, source_edge_hash == output_edge_hash, "Hash struktur edge sama."),
    ("non_hcc_not_valid_community_id", 0, int((account.loc[account["network_position"].eq("HCC"), "community"].astype(str) == "Non-HCC").sum()), int((account.loc[account["network_position"].eq("HCC"), "community"].astype(str) == "Non-HCC").sum()) == 0, "Tidak ada Non-HCC sebagai ID komunitas valid."),
    ("no_duplicate_gephi_id", 0, int(gephi_nodes_out["Id"].duplicated().sum()), int(gephi_nodes_out["Id"].duplicated().sum()) == 0, "Tidak ada duplikasi Id."),
    ("network_position_not_empty", 0, int(gephi_nodes_out["network_position"].isna().sum() + gephi_nodes_out["network_position"].astype(str).str.strip().eq("").sum()), not gephi_nodes_out["network_position"].isna().any() and not gephi_nodes_out["network_position"].astype(str).str.strip().eq("").any(), "Tidak ada network_position kosong."),
    ("rm1_input_hashes_unchanged", input_hashes_before, input_hashes_after, input_hashes_before == input_hashes_after, "File input RM1 hanya dibaca, tidak ditulis ulang."),
    ("sentiment_model_not_rerun", "not invoked", "not invoked", True, "Section ini hanya membaca account_actor_type.csv dan Gephi RM1."),
    ("notebook_run_end_to_end", "final section reached", "final section reached", True, "Jika notebook mencapai cell ini tanpa exception, pipeline end-to-end berhasil."),
    ("png_files_non_empty", len(png_files), sum(path.exists() and path.stat().st_size > 0 for path in png_files), all(path.exists() and path.stat().st_size > 0 for path in png_files), "Semua PNG visualisasi actor type berhasil dibuat dan tidak kosong."),
]
validation_report = pd.DataFrame(validation_rows, columns=["metric", "source_value", "output_value", "passed", "notes"])
validation_report.to_csv(ACTOR_TABLE_DIR / "actor_type_gephi_validation_report.csv", index=False)
if not validation_report["passed"].all():
    failed = validation_report.loc[~validation_report["passed"], ["metric", "source_value", "output_value", "notes"]]
    raise AssertionError("Validasi actor type Gephi gagal:\n" + failed.to_string(index=False))

print("=== RM2 ACTOR TYPE GEPHI EXPORT ===")
print(f"Input LCN nodes       : {LCN_NODES_ACTUAL_PATH.relative_to(ROOT)} ({len(lcn_nodes_source):,} nodes)")
print(f"Input LCN edges       : {LCN_EDGES_ACTUAL_PATH.relative_to(ROOT)} ({len(lcn_edges_source):,} edges)")
print(f"Actor universe        : {len(account):,} accounts")
print("\nActor universe by primary type:")
print(actor_type_universe_summary[["actor_type_primary", "n_accounts", "n_comments"]].to_string(index=False))
print("\nLCN nodes by actor type/network position:")
print(lcn_actor_type_summary[["actor_type_primary", "network_position", "n_nodes", "n_comments"]].to_string(index=False))
print("\nEdge summary by actor-type pair:")
print(edge_summary[["actor_type_pair", "n_edges", "total_weight"]].to_string(index=False))
print("\nOutput Gephi:")
print(f"- {GEPI_NODE_OUT.relative_to(ROOT)}")
print(f"- {GEPI_EDGE_OUT.relative_to(ROOT)}")
print("\nValidasi: semua pemeriksaan actor type Gephi lulus.")


=== RM2 ACTOR TYPE GEPHI EXPORT ===
Input LCN nodes       : output\gephi\gephi_lcn_nodes.csv (724 nodes)
Input LCN edges       : output\gephi\gephi_lcn_edges.csv (1,357 edges)
Actor universe        : 26,427 accounts

Actor universe by primary type:
actor_type_primary  n_accounts  n_comments
  Individual Actor          43        1384
   Community Actor         218        1009
        Mass Actor       26166       31454
             Total       26427       33847

LCN nodes by actor type/network position:
actor_type_primary network_position  n_nodes  n_comments
  Individual Actor              HCC        0           0
  Individual Actor      LCN Non-HCC        0           0
   Community Actor              HCC      218        1009
        Mass Actor      LCN Non-HCC      506        1883

Edge summary by actor-type pair:
      actor_type_pair  n_edges  total_weight
Individual–Individual        0      0.000000
 Individual–Community        0      0.000000
      Individual–Mass        0      0.0

## 33. Panduan Visualisasi Actor Type di Gephi

### Import

Nodes: `output/rm2_actor_type/gephi/gephi_lcn_nodes_actor_type.csv`

Edges: `output/rm2_actor_type/gephi/gephi_lcn_edges_actor_type.csv`

Graph type: `Undirected`

Pastikan tipe kolom:

- `Id`: String
- `Label`: String
- `actor_type_primary`: String
- `network_position`: String
- `community`: String atau Integer sesuai sumber
- `degree`: Integer
- `weighted_degree`: Double
- `betweenness`: Double
- `Weight`: Double

### Tampilan 1 — Actor Type

Partition / Color: `actor_type_primary`

Warna:

- `Individual Actor` = `#E69F00`
- `Community Actor` = `#009E73`
- `Mass Actor` = `#8A8A8A`

Ranking / Size: `weighted_degree`

Rentang awal: minimum node size = `5`, maximum node size = `30`

Edge: color = light gray, opacity = `20–35%`, thickness = `Weight`

Judul: **Struktur Latent Coordination Network Berdasarkan Type Actor**

### Tampilan 2 — Network Position

Partition / Color: `network_position`

Kategori: `HCC`, `LCN Non-HCC`

Judul: **Posisi Aktor dalam Latent Coordination Network**

### Tampilan 3 — Community dan Mass Actor

Filter actor type: `Community Actor`, `Mass Actor`

Gunakan warna actor type dan ukuran `weighted_degree`.

Judul: **Relasi Community Actor dan Mass Actor dalam LCN**

### Tampilan 4 — Edge Pair

Gunakan filter `actor_type_pair`. Tampilkan secara terpisah:

- `Community–Community`
- `Community–Mass`
- `Mass–Mass`
- `Individual–Community`
- `Individual–Mass`

Jangan memakai semua kategori edge dengan warna berbeda sekaligus jika visualisasi menjadi terlalu padat.

### Layout

Urutan layout:

1. OpenOrd
2. ForceAtlas2
3. Noverlap

ForceAtlas2 awal:

- Edge Weight Influence = `1`
- Scaling = `5–15`
- Gravity = `1`
- Prevent Overlap = aktif
- Barnes-Hut Optimization = aktif

Jalankan sampai struktur relatif stabil, bukan sampai komponen terpisah terlalu jauh.

### Batas Interpretasi

Visualisasi hanya mencakup akun yang masuk Latent Coordination Network. Mass Actor di luar LCN tidak ditampilkan karena tidak mempunyai edge koordinasi yang memenuhi kriteria pembentukan jaringan. Seluruh Mass Actor tetap dihitung dalam ringkasan statistik actor universe.

- Individual Actor tidak otomatis merupakan pengendali komunitas.
- Community Actor adalah anggota HCC final, bukan akun yang terbukti sebagai buzzer.
- Mass Actor adalah kategori residual komentator umum, bukan aktor yang terbukti melakukan amplifikasi massal.
- Mass Actor dalam LCN hanya merupakan subset Mass Actor yang mempunyai posisi pada jaringan koordinasi laten.
- Edge menunjukkan hubungan berdasarkan evidence RM1, bukan bukti pembayaran, instruksi, afiliasi, pengaruh kausal, atau astroturfing.
- Node besar menunjukkan weighted degree tinggi, bukan kekuasaan atau kendali.
- Posisi pusat atau betweenness tinggi menunjukkan posisi struktural, bukan kepemimpinan.